# 特徴量インスペクション — keiba-ai-pro

実際に DB から取得した特徴量を**グループ別・列単位**で確認するノートブックです。

| セル | 内容 |
|---|---|
| 1 | パス設定・ライブラリ import |
| 2 | DB 読み込み + 特徴量生成（`add_derived_features`）|
| 3 | 全特徴量サマリー（nan率・型・基本統計）|
| 4 | グループ別詳細インスペクション（ipywidgets ドロップダウン）|
| 5 | 個別列インスペクション（値分布・ヒストグラム）|

In [ ]:
## ── セル 1: パス設定・ライブラリ import ────────────────────────────────
import sys, warnings
from pathlib import Path

# keiba パッケージを参照できるようにワークスペースルートを追加
ROOT = Path("../..").resolve()          # docs/features/ → workspace root
KEIBA = ROOT / "keiba"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(KEIBA))

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import ipywidgets as widgets
from IPython.display import display, HTML

matplotlib.rcParams["font.family"] = "MS Gothic"   # Windows 日本語フォント
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

DB_PATH = KEIBA / "data" / "keiba_ultimate.db"

# ── 期間フィルタ設定（不要なら None のまま）──────────────────────────────
# フォーマット: "YYYY-MM" / "YYYY-MM-DD" / "YYYYMMDD" すべて受け付ける
DATE_START = "2013-01"   # 例: "2013-01"  → 2013年1月以降
DATE_END   = "2018-03"   # 例: "2018-03"  → 2018年3月末まで（月末含む）

print("DB:", DB_PATH, "exists:", DB_PATH.exists())
print("Python:", sys.version.split()[0], "  pandas:", pd.__version__)
print(f"期間フィルタ: {DATE_START or '(開始なし)'} 〜 {DATE_END or '(終了なし)'}")

In [ ]:
## ── セル 2: DB 読み込み + 期間フィルタ + 特徴量生成 ──────────────────────
from keiba_ai.db_ultimate_loader import load_ultimate_training_frame
from keiba_ai.feature_engineering import add_derived_features

def _to_yyyymmdd(s: str) -> str:
    """'YYYY-MM' / 'YYYY-MM-DD' / 'YYYYMMDD' を 'YYYYMMDD' に正規化"""
    s = s.strip().replace("/", "-")
    parts = s.split("-")
    if len(parts) == 2:   # YYYY-MM
        return parts[0] + parts[1].zfill(2) + "01"
    if len(parts) == 3:   # YYYY-MM-DD
        return parts[0] + parts[1].zfill(2) + parts[2].zfill(2)
    return s              # YYYYMMDD そのまま

print("DBロード中...")
df_raw = load_ultimate_training_frame(DB_PATH)
print(f"rawフレーム（全期間）: {len(df_raw):,}行 × {len(df_raw.columns)}列")

# ── 期間フィルタ ──────────────────────────────────────────────────────────
if DATE_START or DATE_END:
    # race_id の先頭 8 桁 = YYYYMMDD で日付判定（race_date より確実）
    race_dates = df_raw["race_id"].astype(str).str[:8]
    mask = pd.Series(True, index=df_raw.index)
    if DATE_START:
        mask &= race_dates >= _to_yyyymmdd(DATE_START)
    if DATE_END:
        _end_raw = _to_yyyymmdd(DATE_END)
        # YYYY-MM 指定の場合は月末まで含める（翌月初日 "未満" に変換）
        if _end_raw[6:] == "01" and len(DATE_END.replace("/", "-").split("-")) == 2:
            from datetime import date as _date
            y, m = int(_end_raw[:4]), int(_end_raw[4:6])
            _next = _date(y + (m // 12), (m % 12) + 1, 1) if m < 12 else _date(y + 1, 1, 1)
            _end_raw = _next.strftime("%Y%m%d")
        mask &= race_dates < _end_raw
    df_raw = df_raw[mask].copy()
    _date_range = f"{DATE_START or '先頭'} 〜 {DATE_END or '末尾'}"
    print(f"期間フィルタ適用後: {len(df_raw):,}行  ({_date_range})")
    print(f"  最古 race_id: {df_raw['race_id'].min()}  最新 race_id: {df_raw['race_id'].max()}")

print("\n特徴量生成中（add_derived_features）...")
df = add_derived_features(df_raw)
print(f"完成フレーム: {len(df):,}行 × {len(df.columns)}列")

In [ ]:
## ── セル 3: 全特徴量サマリー ────────────────────────────────────────────
# 各列の型・nan率・ユニーク数・代表値をまとめた一覧表を生成する

def _safe_nunique(s):
    """list/dict 等の unhashable 型を含む列でも安全にユニーク数を返す"""
    try:
        return s.nunique(dropna=True)
    except TypeError:
        return -1  # unhashable

def _safe_top_values(s):
    """value_counts が使えない列は代替テキストを返す"""
    try:
        top = s.value_counts(dropna=True).head(3)
        return " | ".join(f"{k}:{v}" for k, v in top.items())
    except TypeError:
        return "(list/dict 型 — 集計不可)"

rows = []
for col in df.columns:
    s = df[col]
    nan_pct = s.isna().mean() * 100
    nuniq   = _safe_nunique(s)
    dtype   = str(s.dtype)
    if pd.api.types.is_numeric_dtype(s):
        try:
            q = s.quantile([0.0, 0.25, 0.5, 0.75, 1.0])
            r = {"min": round(q[0.0], 3), "Q1": round(q[0.25], 3),
                 "median": round(q[0.5], 3), "Q3": round(q[0.75], 3),
                 "max": round(q[1.0], 3), "top_values": ""}
        except Exception:
            r = {"min": "", "Q1": "", "median": "", "Q3": "", "max": "", "top_values": ""}
    else:
        r = {"min": "", "Q1": "", "median": "", "Q3": "", "max": "",
             "top_values": _safe_top_values(s)}
    rows.append({"column": col, "dtype": dtype, "nan_%": round(nan_pct, 1), "unique": nuniq, **r})

summary_df = pd.DataFrame(rows)
print(f"合計 {len(summary_df)} 列")
display(summary_df.style.background_gradient(subset=["nan_%"], cmap="Reds"))

In [ ]:
## ── セル 4: グループ別詳細インスペクション ──────────────────────────────
# 特徴量をグループに分類して、ドロップダウンで切り替えて確認できる

FEATURE_GROUPS = {
    "01_レース基本情報 (_fe_id_season)": [
        "race_class_num", "venue_code", "race_num", "n_horses",
        "cos_date", "sin_date", "season",
        "sex_code", "seasonal_sex", "frame_race_type",
        "kai_num", "kai_cos", "kai_sin",
        "day_num", "day_cos", "day_sin", "is_late_opening",
        "is_female_only_race", "is_3yo_limited",
        "race_age_condition", "class_rank_adj", "weight_vs_standard",
    ],
    "02_コース特性 (_fe_course)": [
        "straight_length", "track_type", "corner_radius",
        "inner_bias", "inner_advantage",
    ],
    "03_馬属性・休養 (_fe_horse_category)": [
        "is_young", "is_prime", "is_veteran",
        "running_style_num", "rest_category",
        "rest_short", "rest_normal", "rest_long", "rest_very_long",
    ],
    "04_オッズ市場 (_fe_market)": [
        "implied_prob", "log_odds", "odds_is_missing",
        "implied_prob_norm", "odds_rank_in_race", "odds_z_in_race",
        "market_entropy", "top3_probability",
        "popularity_normalized",
    ],
    "05_前走・近走情報 (_fe_prev_race)": [
        "days_since_last_race",
        "distance_change", "distance_increased", "distance_decreased",
        "horse_win_rate",
        "prev_speed_index", "prev_speed_zscore",
        "prev2_speed_index", "prev2_speed_zscore",
        "speed_index_change", "speed_avg_weighted", "speed_best_2",
        "speed_improving",
        "recent_form_weighted", "form_trend",
        "recent_form_5race", "form_consistency",
        "win_count_5", "top3_count_5",
        "speed_best_5", "speed_trend_5",
        "prev_race_class_num", "class_change",
        "is_class_up", "is_class_down", "is_same_class", "class_diff_abs",
    ],
    "06_相手関係 (_fe_opponent)": [
        "race_avg_prev_speed", "race_max_prev_speed",
        "speed_vs_race_avg", "horse_speed_rank_pct",
        "race_avg_prev_finish",
    ],
    "07_持ちタイム (_fe_holding_time)": [
        "has_just_data", "holding_just_speed",
        "holding_just_time_rank", "holding_just_finish_rank",
        "holding_just_babasa", "holding_short_babasa",
        "holding_middle_babasa", "holding_long_babasa",
    ],
    "08_ラップ展開 (_fe_lap)": [
        "lap_200m", "lap_400m", "lap_600m", "lap_800m",
        "lap_1000m", "lap_1200m", "lap_1600m", "lap_2000m",
        "lap_sect_200m", "lap_sect_400m", "lap_sect_600m",
        "lap_sect_800m", "lap_sect_1000m", "lap_sect_1200m",
        "race_pace_front", "race_pace_back",
        "race_pace_diff", "race_pace_ratio",
    ],
    "09_脚質 (_fe_corner_position)": [
        "corner_first", "corner_last", "corner_gain",
        "running_style_code",
    ],
    "10_速度指数テーブル (_fe_speed_figures)": [
        "sf_index_last", "sf_index_2ago", "sf_index_3ago",
        "sf_max_index", "sf_course_max_index", "sf_dist_max_index",
        "sf_index_trend",
    ],
    "11_調教 (_fe_training)": [
        "last_training_time_3f", "last_training_grade_encoded",
        "has_training_data", "training_comment_score",
    ],
    "12_通算・近走統計 (_fe_history)": [
        "horse_win_rate_exp", "horse_show_rate_exp",
        "jockey_win_rate", "jockey_show_rate",
        "trainer_win_rate", "trainer_show_rate",
        "sire_win_rate", "sire_show_rate",
        "horse_surface_win_rate", "horse_surface_avg_finish",
        "horse_distance_win_rate", "horse_distance_avg_finish",
        "horse_avg_class_num", "class_drop",
        "gate_bracket_win_rate",
        "jockey_course_win_rate",
        "sire_surface_win_rate", "sire_dist_band_win_rate",
        "running_style_mean_5", "running_style_std_5",
        "horse_recent30_win_rate",
        "horse_speed_exp_mean", "horse_speed_exp_std",
    ],
    "13_欠損フラグ (_fe_missing_flags)": [
        c for c in df.columns if c.endswith("_is_missing")
    ],
}

def inspect_group(group_name):
    cols = [c for c in FEATURE_GROUPS.get(group_name, []) if c in df.columns]
    missing = [c for c in FEATURE_GROUPS.get(group_name, []) if c not in df.columns]
    if missing:
        print(f"⚠ 未生成列 ({len(missing)}): {missing}")
    if not cols:
        print("表示できる列がありません")
        return
    rows = []
    for col in cols:
        s = df[col]
        nan_pct = s.isna().mean() * 100
        zero_pct = (s == 0).mean() * 100 if pd.api.types.is_numeric_dtype(s) else 0
        vc = s.value_counts(dropna=True).head(5)
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "nan_%": round(nan_pct, 1),
            "zero_%": round(zero_pct, 1) if pd.api.types.is_numeric_dtype(s) else "",
            "unique": s.nunique(dropna=True),
            "mean": round(s.mean(), 4) if pd.api.types.is_numeric_dtype(s) else "",
            "std":  round(s.std(),  4) if pd.api.types.is_numeric_dtype(s) else "",
            "top5_values": " | ".join(f"{k}:{v}" for k, v in vc.items()),
        })
    grp_df = pd.DataFrame(rows)
    display(grp_df.style
            .background_gradient(subset=["nan_%"], cmap="Oranges")
            .set_caption(group_name))

dd = widgets.Dropdown(
    options=list(FEATURE_GROUPS.keys()),
    description="グループ:",
    layout=widgets.Layout(width="60%"),
)
out = widgets.Output()

def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        with out:
            out.clear_output()
            inspect_group(change["new"])

dd.observe(on_change)
display(dd, out)
# 初期表示
with out:
    inspect_group(dd.value)

In [ ]:
## ── セル 5: 個別列インスペクション（ヒストグラム + 統計）────────────────
# 列名を選択して値の分布・統計を確認する

all_cols = sorted(df.columns.tolist())

col_selector = widgets.Combobox(
    placeholder="列名を入力または選択",
    options=all_cols,
    description="列:",
    layout=widgets.Layout(width="60%"),
    ensure_option=False,
)
out2 = widgets.Output()

def show_column(col_name):
    out2.clear_output()
    with out2:
        if col_name not in df.columns:
            print(f"'{col_name}' は存在しません")
            return
        s = df[col_name]
        nan_n   = s.isna().sum()
        valid_n = s.notna().sum()
        print(f"{'='*55}")
        print(f"列名    : {col_name}")
        print(f"dtype   : {s.dtype}")
        print(f"行数    : {len(s):,}  有効: {valid_n:,}  NaN: {nan_n:,} ({nan_n/len(s)*100:.1f}%)")
        print(f"unique  : {s.nunique(dropna=True):,}")

        if pd.api.types.is_numeric_dtype(s):
            desc = s.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
            print(desc.to_string())
            # ヒストグラム
            fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
            s_valid = s.dropna()
            axes[0].hist(s_valid, bins=50, color="#4C78A8", edgecolor="white", linewidth=0.3)
            axes[0].set_title(f"{col_name}  (histogram)")
            axes[0].set_xlabel("value")
            axes[0].set_ylabel("count")
            # 上位20カテゴリ（整数値の場合）
            vc = s.value_counts(dropna=True).head(20)
            axes[1].barh(vc.index.astype(str)[::-1], vc.values[::-1], color="#72B7B2")
            axes[1].set_title(f"top-20 values")
            axes[1].set_xlabel("count")
            plt.tight_layout()
            plt.show()
        else:
            vc = s.value_counts(dropna=False).head(20)
            print("\nvalue_counts (top 20):")
            print(vc.to_string())
            fig, ax = plt.subplots(figsize=(10, max(3, len(vc) * 0.35)))
            ax.barh(vc.index.astype(str)[::-1], vc.values[::-1], color="#54A24B")
            ax.set_title(f"{col_name}  top-20 values")
            ax.set_xlabel("count")
            plt.tight_layout()
            plt.show()

def on_col_change(change):
    if change["type"] == "change" and change["name"] == "value":
        show_column(change["new"])

col_selector.observe(on_col_change)
display(col_selector, out2)
# 初期表示
with out2:
    show_column("race_class_num")

In [ ]:
## ── セル 6: nan率ランキング（高いほど注意が必要な列）─────────────────
nan_rates = df.isna().mean().sort_values(ascending=False)
nan_top = nan_rates[nan_rates > 0.01].reset_index()
nan_top.columns = ["column", "nan_rate"]
nan_top["nan_%"] = (nan_top["nan_rate"] * 100).round(1)
nan_top["dtype"] = nan_top["column"].map(lambda c: str(df[c].dtype))

fig, ax = plt.subplots(figsize=(10, max(4, len(nan_top) * 0.22)))
colors = ["#E45756" if r > 0.5 else "#F58518" if r > 0.2 else "#EECA3B"
          for r in nan_top["nan_rate"]]
ax.barh(nan_top["column"][::-1], nan_top["nan_%"][::-1], color=colors[::-1])
ax.axvline(20, color="orange", linestyle="--", linewidth=0.8, label="20%")
ax.axvline(50, color="red",    linestyle="--", linewidth=0.8, label="50%")
ax.set_xlabel("NaN率 (%)")
ax.set_title(f"NaN率 > 1% の列一覧 ({len(nan_top)}列)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nNaN = 0%  : {(nan_rates == 0).sum()}列")
print(f"NaN 1-20% : {((nan_rates > 0.01) & (nan_rates <= 0.2)).sum()}列")
print(f"NaN 20-50%: {((nan_rates > 0.2)  & (nan_rates <= 0.5)).sum()}列")
print(f"NaN > 50% : {(nan_rates > 0.5).sum()}列")

In [ ]:
## ── セル 7: prepare_for_lightgbm_ultimate を適用して前処理後データを取得 ─
from keiba_ai.lightgbm_feature_optimizer import prepare_for_lightgbm_ultimate

# ── 学習ターゲット列（1つだけコメントを外して使用）────────────────────────
# TARGET = "win"           # 1着 (finish == 1) → 単勝予測
# TARGET = "place3"      # 3着以内 (finish <= 3) → 複勝予測
# TARGET = "win_tie"     # 1着 + 同タイム同着馬を正例 → win の厳密版
TARGET = "speed_deviation"  # レース内速度 z-score → 回帰ターゲット
# TARGET = "rank"        # 着順スコア（LGBMRanker 用 label_gain）

print(f"前処理前: {df.shape[0]:,}行 × {df.shape[1]:,}列  target={TARGET!r}")

df_pre, optimizer, cat_features = prepare_for_lightgbm_ultimate(
    df.copy(),
    target_col=TARGET,
    is_training=True,
)

print(f"前処理後: {df_pre.shape[0]:,}行 × {df_pre.shape[1]:,}列")
print(f"カテゴリカル特徴量 ({len(cat_features)}): {cat_features}")

In [ ]:
## ── セル 8: 前処理前後の列比較（追加・削除された列を確認）─────────────
cols_before = set(df.columns)
cols_after  = set(df_pre.columns)

dropped  = sorted(cols_before - cols_after)
added    = sorted(cols_after  - cols_before)
kept     = sorted(cols_before & cols_after)

print(f"削除された列: {len(dropped)}")
print(f"追加された列: {len(added)}")
print(f"残存した列:   {len(kept)}")
print()

from keiba_ai.constants import UNNECESSARY_COLUMNS, FUTURE_FIELDS

unnecessary        = [c for c in dropped if c in UNNECESSARY_COLUMNS]
future             = [c for c in dropped if c in FUTURE_FIELDS]
obj_type           = [c for c in dropped if df[c].dtype == object
                      and c not in unnecessary and c not in future]
optimizer_excluded = [c for c in dropped if c not in unnecessary
                      and c not in future and c not in obj_type]

print(f"  UNNECESSARY_COLUMNS : {len(unnecessary)}")
print(f"  FUTURE_FIELDS       : {len(future)}")
print(f"  object型除外        : {len(obj_type)}")
print(f"  optimizer除外       : {len(optimizer_excluded)}")
print()

# nan率を一括計算（isna() だけなら list 型でも高速）
def _fast_nan_pct(col):
    try:
        arr = df[col].values
        nan_count = sum(1 for v in arr if v is None or (not isinstance(v, (list, dict)) and pd.isna(v)))
        return nan_count / len(arr) * 100
    except Exception:
        return float("nan")

compare_rows = []
for c in dropped:
    nan_pct = _fast_nan_pct(c)
    reason = (
        "UNNECESSARY_COLUMNS" if c in UNNECESSARY_COLUMNS
        else "FUTURE_FIELDS" if c in FUTURE_FIELDS
        else "object型除外" if df[c].dtype == object
        else "optimizer除外"
    )
    compare_rows.append({"column": c, "dtype": str(df[c].dtype),
                         "nan_%": round(nan_pct, 1), "reason": reason})

compare_df = pd.DataFrame(compare_rows)
display(compare_df.style
        .applymap(lambda v: "background-color:#3a1a1a" if v == "FUTURE_FIELDS"
                  else ("background-color:#1a2a3a" if v == "UNNECESSARY_COLUMNS"
                        else ""), subset=["reason"])
        .set_caption(f"削除列一覧 ({len(dropped)}列)")
        .background_gradient(subset=["nan_%"], cmap="Reds"))

In [ ]:
## ── セル 9: 前処理後データのサマリー（モデル入力列の全体像）────────────
pre_rows = []
for col in df_pre.columns:
    s = df_pre[col]
    nan_pct = s.isna().mean() * 100
    try:
        nuniq = s.nunique(dropna=True)
    except TypeError:
        nuniq = -1
    is_cat = col in cat_features
    if pd.api.types.is_numeric_dtype(s):
        try:
            q = s.quantile([0.0, 0.25, 0.5, 0.75, 1.0])
            r = {"min": round(q[0.0], 3), "Q1": round(q[0.25], 3),
                 "median": round(q[0.5], 3), "Q3": round(q[0.75], 3),
                 "max": round(q[1.0], 3)}
        except Exception:
            r = {"min": "", "Q1": "", "median": "", "Q3": "", "max": ""}
    else:
        r = {"min": "", "Q1": "", "median": "", "Q3": "", "max": ""}
    pre_rows.append({
        "column": col, "dtype": str(s.dtype),
        "is_categorical": "✓" if is_cat else "",
        "nan_%": round(nan_pct, 1), "unique": nuniq,
        **r,
    })

pre_summary = pd.DataFrame(pre_rows)
print(f"前処理後: 合計 {len(pre_summary)} 列（うちカテゴリカル {len(cat_features)} 列）")
display(pre_summary.style.background_gradient(subset=["nan_%"], cmap="Oranges"))

In [ ]:
## ── セル 10: 前処理後の NaN 率ランキング ─────────────────────────────────
pre_nan_rates = df_pre.isna().mean().sort_values(ascending=False)
pre_nan_top   = pre_nan_rates[pre_nan_rates > 0.01].reset_index()
pre_nan_top.columns = ["column", "nan_rate"]
pre_nan_top["nan_%"] = (pre_nan_top["nan_rate"] * 100).round(1)

# 前処理後のNaN率ランキング
fig, axes = plt.subplots(1, 2, figsize=(16, max(4, len(pre_nan_top) * 0.2)))

# 左: 前処理後バーチャート
colors_pre = ["#E45756" if r > 0.5 else "#F58518" if r > 0.2 else "#EECA3B"
              for r in pre_nan_top["nan_rate"]]
axes[0].barh(pre_nan_top["column"][::-1], pre_nan_top["nan_%"][::-1], color=colors_pre[::-1])
axes[0].axvline(20, color="orange", linestyle="--", linewidth=0.8, label="20%")
axes[0].axvline(50, color="red",    linestyle="--", linewidth=0.8, label="50%")
axes[0].set_xlabel("NaN率 (%)")
axes[0].set_title(f"前処理後 NaN率 > 1% の列 ({len(pre_nan_top)}列)")
axes[0].legend()

# 右: 前処理前後の NaN率分布比較（積み上げ棒グラフ）
before_bins = [0, 1, 20, 50, 100]
before_labels = ["0%", "1-20%", "20-50%", ">50%"]
before_counts = [
    (nan_rates == 0).sum(),
    ((nan_rates > 0.01) & (nan_rates <= 0.2)).sum(),
    ((nan_rates > 0.2)  & (nan_rates <= 0.5)).sum(),
    (nan_rates > 0.5).sum(),
]
after_counts = [
    (pre_nan_rates == 0).sum(),
    ((pre_nan_rates > 0.01) & (pre_nan_rates <= 0.2)).sum(),
    ((pre_nan_rates > 0.2)  & (pre_nan_rates <= 0.5)).sum(),
    (pre_nan_rates > 0.5).sum(),
]
x = range(len(before_labels))
width = 0.35
bars1 = axes[1].bar([i - width/2 for i in x], before_counts, width, label="前処理前", color="#4C78A8")
bars2 = axes[1].bar([i + width/2 for i in x], after_counts,  width, label="前処理後", color="#72B7B2")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(before_labels)
axes[1].set_ylabel("列数")
axes[1].set_title("前処理前後 NaN率分布比較")
axes[1].legend()
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    if h > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.5, str(int(h)),
                     ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

print(f"前処理後 NaN = 0%  : {(pre_nan_rates == 0).sum()}列")
print(f"前処理後 NaN 1-20% : {((pre_nan_rates > 0.01) & (pre_nan_rates <= 0.2)).sum()}列")
print(f"前処理後 NaN 20-50%: {((pre_nan_rates > 0.2)  & (pre_nan_rates <= 0.5)).sum()}列")
print(f"前処理後 NaN > 50% : {(pre_nan_rates > 0.5).sum()}列")

In [ ]:
## ── セル 11: 前処理後の個別列インスペクション ────────────────────────────
pre_cols = sorted(df_pre.columns.tolist())

pre_col_selector = widgets.Combobox(
    placeholder="前処理後の列名を入力または選択",
    options=pre_cols,
    description="列 (前処理後):",
    layout=widgets.Layout(width="65%"),
    ensure_option=False,
)
pre_out = widgets.Output()

def show_pre_column(col_name):
    pre_out.clear_output()
    with pre_out:
        if col_name not in df_pre.columns:
            print(f"'{col_name}' は前処理後データに存在しません")
            return
        s     = df_pre[col_name]
        s_raw = df[col_name] if col_name in df.columns else None
        nan_n   = s.isna().sum()
        valid_n = s.notna().sum()
        is_cat  = col_name in cat_features
        print(f"{'='*55}")
        print(f"列名    : {col_name}  {'[CATEGORICAL]' if is_cat else ''}")
        print(f"dtype   : {s.dtype}")
        print(f"行数    : {len(s):,}  有効: {valid_n:,}  NaN: {nan_n:,} ({nan_n/len(s)*100:.1f}%)")
        print(f"unique  : {s.nunique(dropna=True):,}")
        if s_raw is None:
            print("(前処理前データに同名列なし)")

        if pd.api.types.is_numeric_dtype(s):
            print(s.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_string())
            ncols = 2 if s_raw is not None and pd.api.types.is_numeric_dtype(s_raw) else 1
            fig, axes = plt.subplots(1, ncols, figsize=(12 if ncols == 2 else 8, 3.5))
            if ncols == 1:
                axes = [axes]
            axes[0].hist(s.dropna(), bins=50, color="#72B7B2", edgecolor="white", linewidth=0.3)
            axes[0].set_title(f"{col_name} — 前処理後")
            axes[0].set_xlabel("value"); axes[0].set_ylabel("count")
            if ncols == 2:
                axes[1].hist(s_raw.dropna(), bins=50, color="#4C78A8", edgecolor="white", linewidth=0.3)
                axes[1].set_title(f"{col_name} — 前処理前 (raw)")
                axes[1].set_xlabel("value"); axes[1].set_ylabel("count")
            plt.tight_layout()
            plt.show()
        else:
            vc = s.value_counts(dropna=False).head(20)
            print("\nvalue_counts (top 20):"); print(vc.to_string())
            fig, ax = plt.subplots(figsize=(10, max(3, len(vc) * 0.35)))
            ax.barh(vc.index.astype(str)[::-1], vc.values[::-1], color="#54A24B")
            ax.set_title(f"{col_name}  top-20 (前処理後)"); ax.set_xlabel("count")
            plt.tight_layout(); plt.show()

def on_pre_col_change(change):
    if change["type"] == "change" and change["name"] == "value":
        show_pre_column(change["new"])

pre_col_selector.observe(on_pre_col_change)
display(pre_col_selector, pre_out)
with pre_out:
    show_pre_column(pre_cols[0] if pre_cols else "")

# 欠損値断捨離レポート

パイプライン全体（DB取得 → 特徴量エンジニアリング → 前処理）の欠損値を分類し、
**データ再取得・前処理補完・削除** のどれで対応すべきかを整理します。

| アクション | 意味 |
|---|---|
| **そのまま** | 設計上の欠損（初走など）。欠損フラグで対応済 |
| **前処理補完** | 中央値・0・グループ平均などで埋められる |
| **再スクレイプ** | データは存在するが収集できていない |
| **削除推奨** | NaN 率が高すぎる or 対象期間では意味をなさない |
| **要確認** | 個別判断が必要 |

In [ ]:
## ── セル 12: 欠損値断捨離レポート ─────────────────────────────────────

# ── 欠損原因・推奨アクションの分類ルール ─────────────────────────────────
# 優先度: 高=今すぐ対処 / 中=次イテレーション / 低=後回し可 / -=対処不要

def _classify(col: str, nan_pct: float) -> tuple:
    """(欠損原因, 推奨アクション, 優先度) を返す"""
    c = col

    # ── 欠損フラグ列（0/1 なのに NaN は異常）──────────────────────────────
    if c.endswith("_is_missing"):
        return ("欠損フラグ列がNaN", "前処理: fillna(0)", "高") if nan_pct > 5 \
               else ("欠損フラグ正常", "そのまま", "-")

    # ── ラップ・ペース（旧データ形式で欠落）────────────────────────────────
    if c.startswith(("lap_", "race_pace_")):
        if nan_pct > 70:
            return ("スクレイプ未収集\n旧データ形式欠落", "削除推奨\n(高NaN期間)", "高")
        return ("スクレイプ未収集", "再スクレイプで改善", "中")

    # ── 持ちタイム（just時間テーブル）──────────────────────────────────────
    if c.startswith("holding_") or c == "has_just_data":
        if nan_pct > 60:
            return ("スクレイプ未収集\n(just時間テーブル)", "削除推奨 or\n2019年以降限定", "高")
        return ("スクレイプ未収集", "再スクレイプで改善", "中")

    # ── 速度指数テーブル（sf_*）────────────────────────────────────────────
    if c.startswith("sf_"):
        if nan_pct > 60:
            return ("スクレイプ未収集\n(速度指数テーブル)", "削除推奨 or\n2019年以降限定", "高")
        return ("スクレイプ未収集", "再スクレイプで改善", "中")

    # ── 調教データ──────────────────────────────────────────────────────────
    if c.startswith(("last_training_", "training_")) or c == "has_training_data":
        return ("スクレイプ未収集\n(調教データ)", "削除推奨 or 再スクレイプ", "中")

    # ── コーナー位置・脚質（レース結果ページ依存）──────────────────────────
    if c in ("corner_first", "corner_last", "corner_gain", "running_style_code",
             "running_style_num"):
        return ("スクレイプ部分欠損\n(コーナー位置)", "前処理: 最頻値補完 or 削除", "中")

    # ── 前走・近走系（初走=構造的欠損）────────────────────────────────────
    structural_prefixes = (
        "prev_", "prev2_", "days_since_last_race",
        "distance_change", "distance_increased", "distance_decreased",
        "speed_index_change", "speed_avg_", "speed_best_", "speed_trend_",
        "speed_improving",
        "recent_form_", "form_", "win_count_", "top3_count_",
        "running_style_mean_", "running_style_std_",
        "is_class_up", "is_class_down", "is_same_class", "class_diff_abs",
        "class_change", "prev_race_class_num",
    )
    if c.startswith(structural_prefixes) or c in structural_prefixes:
        return ("構造的欠損\n(初走=前走なし)", "そのまま\n(欠損フラグ対応済)", "-")

    # ── 通算成績（expanding window、初走で0件）──────────────────────────────
    if c.startswith(("horse_win_rate", "horse_show_rate", "horse_avg_",
                      "horse_surface_", "horse_distance_", "horse_venue_",
                      "horse_recent", "horse_speed_exp")):
        return ("構造的欠損\n(初出場=履歴なし)", "ベイズ平滑化 or 0補完", "低")

    # ── 騎手・調教師統計──────────────────────────────────────────────────
    if c.startswith(("jockey_", "trainer_")):
        if nan_pct > 30:
            return ("統計計算不足\n(初出場騎手多)", "ベイズ平滑化 / 全体平均補完", "中")
        return ("統計計算不足\n(初出場)", "そのまま or 補完", "低")

    # ── 父馬・母父統計──────────────────────────────────────────────────────
    if c.startswith(("sire_", "damsire_")):
        return ("統計計算不足\n(父馬データ不足)", "ベイズ平滑化 / 全体平均補完", "低")

    # ── オッズ・人気系──────────────────────────────────────────────────────
    if c in ("odds", "popularity", "implied_prob", "log_odds", "odds_z_in_race",
             "market_entropy", "top3_probability", "popularity_normalized",
             "implied_prob_norm", "odds_rank_in_race", "log_odds"):
        if nan_pct > 20:
            return ("スクレイプ欠損\n(オッズ未収集)", "再スクレイプ or 削除", "高")
        return ("スクレイプ部分欠損", "前処理: 中央値補完", "低")

    # ── 馬体重──────────────────────────────────────────────────────────────
    if "horse_weight" in c or c == "weight_vs_standard":
        return ("スクレイプ部分欠損\n(馬体重未報告)", "前処理: 前走体重で補完", "低")

    # ── NaN ほぼなし（正常）────────────────────────────────────────────────
    if nan_pct < 2:
        return ("正常 (低NaN)", "そのまま", "-")

    return ("不明 / 要確認", "個別確認", "中")


# ── 全列の分類テーブルを作成 ────────────────────────────────────────────
# グループ名を逆引きマップ化
col_to_group = {}
for grp, cols in FEATURE_GROUPS.items():
    for c in cols:
        col_to_group[c] = grp

report_rows = []
for col in sorted(df.columns):
    nr = nan_rates.get(col, 0.0) * 100        # nan_rates は 0-1 スケール
    pre_nr = pre_nan_rates.get(col, None)
    in_model = col in df_pre.columns

    cause, action, priority = _classify(col, nr)
    report_rows.append({
        "column":       col,
        "group":        col_to_group.get(col, "(その他)"),
        "nan_%_before": round(nr, 1),
        "nan_%_after":  round(pre_nr * 100, 1) if (in_model and pre_nr is not None) else ("—" if in_model else "除外"),
        "in_model":     "✓" if in_model else "",
        "欠損原因":     cause,
        "推奨アクション": action,
        "優先度":       priority,
    })

report_df = pd.DataFrame(report_rows)

# 優先度でソート（高→中→低→-）
priority_order = {"高": 0, "中": 1, "低": 2, "-": 3}
report_df["_sort"] = report_df["優先度"].map(priority_order)
report_df = report_df.sort_values(["_sort", "nan_%_before"], ascending=[True, False]).drop(columns="_sort")

# サマリー
print(f"=== 断捨離サマリー  期間: {DATE_START} 〜 {DATE_END}  対象列数: {len(report_df)} ===")
print()
for action_grp, sub in report_df.groupby("推奨アクション", sort=False):
    print(f"  {action_grp:30s}: {len(sub):3d}列  (NaN>50%: {(sub['nan_%_before']>50).sum():2d}列)")
print()
print(f"  モデル入力列 (in_model=✓): {report_df['in_model'].eq('✓').sum()}列")

# 優先度「高」の列を強調表示
high_df = report_df[report_df["優先度"] == "高"][
    ["column", "group", "nan_%_before", "欠損原因", "推奨アクション"]
]
if not high_df.empty:
    print(f"\n【優先度: 高】 {len(high_df)}列 — 今すぐ対処推奨")
    display(high_df.style
            .background_gradient(subset=["nan_%_before"], cmap="Reds")
            .set_caption("優先度: 高"))

# 全体テーブル（折りたたみ可能な長さなのでそのまま表示）
print(f"\n【全列一覧】")
display(report_df.drop(columns=["nan_%_after"]).style
        .applymap(lambda v: "background-color:#3a1a1a" if v == "高"
                  else ("background-color:#2a3a1a" if v == "低" else ""), subset=["優先度"])
        .background_gradient(subset=["nan_%_before"], cmap="Reds")
        .set_caption(f"特徴量断捨離レポート  {DATE_START}〜{DATE_END}"))

In [ ]:
## ── セル 13: 断捨離サマリー可視化 ──────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── 左: アクション別・列数の棒グラフ ────────────────────────────────────
action_counts = report_df.groupby("推奨アクション").size().sort_values(ascending=False)
color_map = {
    "削除推奨 or\n(高NaN期間)": "#E45756",
    "削除推奨\n(高NaN期間)": "#E45756",
    "削除推奨 or 再スクレイプ": "#F58518",
    "削除推奨 or\n2019年以降限定": "#F58518",
    "削除推奨\n(高NaN期間)": "#E45756",
    "再スクレイプで改善": "#EECA3B",
    "再スクレイプで改善\n": "#EECA3B",
    "そのまま\n(欠損フラグ対応済)": "#54A24B",
    "そのまま": "#54A24B",
    "前処理: fillna(0)": "#72B7B2",
    "前処理: 最頻値補完 or 削除": "#72B7B2",
    "前処理: 中央値補完": "#72B7B2",
    "前処理: 前走体重で補完": "#72B7B2",
    "ベイズ平滑化 / 全体平均補完": "#B279A2",
    "ベイズ平滑化 or 0補完": "#B279A2",
    "そのまま or 補完": "#9ECAE1",
    "欠損フラグ正常": "#54A24B",
    "個別確認": "#BAB0AC",
}
bar_colors = [color_map.get(a, "#BAB0AC") for a in action_counts.index]
axes[0].barh(action_counts.index[::-1], action_counts.values[::-1], color=bar_colors[::-1])
axes[0].set_xlabel("列数")
axes[0].set_title("推奨アクション別 列数")
for i, v in enumerate(action_counts.values[::-1]):
    axes[0].text(v + 0.3, i, str(v), va="center", fontsize=8)

# ── 中: グループ別 平均NaN率 ────────────────────────────────────────────
grp_nan = (report_df.groupby("group")["nan_%_before"]
           .mean().sort_values(ascending=False))
grp_short = {g: g.split("(")[0][:25].strip() for g in grp_nan.index}
axes[1].barh(
    [grp_short.get(g, g) for g in grp_nan.index[::-1]],
    grp_nan.values[::-1],
    color=["#E45756" if v > 50 else "#F58518" if v > 20 else "#EECA3B" if v > 5 else "#54A24B"
           for v in grp_nan.values[::-1]],
)
axes[1].axvline(20, color="orange", linestyle="--", linewidth=0.8)
axes[1].axvline(50, color="red",    linestyle="--", linewidth=0.8)
axes[1].set_xlabel("平均NaN率 (%)")
axes[1].set_title("グループ別 平均NaN率")

# ── 右: 優先度別パイチャート ─────────────────────────────────────────────
priority_counts = report_df["優先度"].value_counts()
pie_colors = {"高": "#E45756", "中": "#F58518", "低": "#EECA3B", "-": "#54A24B"}
axes[2].pie(
    priority_counts.values,
    labels=[f"{k} ({v}列)" for k, v in priority_counts.items()],
    colors=[pie_colors.get(k, "#BAB0AC") for k in priority_counts.index],
    autopct="%1.0f%%",
    startangle=90,
)
axes[2].set_title("優先度分布")

plt.suptitle(
    f"特徴量断捨離レポート  期間: {DATE_START} 〜 {DATE_END}  ({len(report_df)}列)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

# ── テキストサマリー ─────────────────────────────────────────────────────
print("=" * 65)
print("【断捨離方針まとめ】")
print()
print("■ 削除推奨（高NaN / 対象期間では意味をなさない）")
del_cols = report_df[report_df["推奨アクション"].str.startswith("削除")]["column"].tolist()
print(f"  {del_cols}")
print()
print("■ 再スクレイプで改善可能")
rescrape_cols = report_df[report_df["推奨アクション"].str.startswith("再スクレイプ")]["column"].tolist()
print(f"  {rescrape_cols}")
print()
print("■ 前処理補完で対応可能")
impute_cols = report_df[report_df["推奨アクション"].str.startswith("前処理") |
                         report_df["推奨アクション"].str.startswith("ベイズ")]["column"].tolist()
print(f"  {impute_cols}")
print()
print("■ そのまま（設計上の欠損 / 低NaN）")
ok_cols = report_df[report_df["優先度"] == "-"]["column"].tolist()
print(f"  {len(ok_cols)}列 — 特に対処不要")

# LightGBM 学習実行セクション

## 設計方針

| 項目 | 採用方針 | 理由 |
|---|---|---|
| **データソース** | 上セル（Cell 2/7）の `df`, `df_pre` をそのまま使用 | DB再ロード不要・高速 |
| **学習コード** | `lgb.train()` を直接呼び出し（本番 `routers/train.py` と同一パス）| sklearn Pipeline を経由せず本番と完全一致 |
| **ターゲット** | `_make_target(df, TRAIN_TARGET)` — `df` から毎回生成 | `df_pre` にターゲット列は含まれない |
| **前処理** | `df_pre` / `cat_features` を再利用（`TRAIN_TARGET` が変わった場合のみ再実行） | 冪等 |
| **Optuna** | `USE_OPTUNA=True` 時のみ `OptunaLightGBMOptimizer.optimize()` を実行 | デフォルト OFF で高速起動 |
| **モデル保存** | `keiba/data/models/model_{target}_{ts}.joblib` — 本番バンドル形式 | `predict.py` が直接ロード可能 |

## 実行フロー

```
Cell E（設定）  →  Cell F（データ準備）  →  Cell G（学習 / Optuna）  →  Cell H（保存）
```

> **Note**: Cell G の先頭で `USE_OPTUNA` を確認します。`True` の場合は Optuna 最適化 → 最終モデル再学習、`False` の場合はデフォルトパラメータで直接学習します。

In [ ]:
## ── Cell E: 学習設定 ── ここを編集して実行 ──────────────────────────────

# ── ターゲット ────────────────────────────────────────────────────────────
# Cell 7 の TARGET と一致させると df_pre を再利用できる（高速）
TRAIN_TARGET = "speed_deviation"   # win / place3 / win_tie / speed_deviation / rank

# ── LightGBM デフォルトパラメータ（USE_OPTUNA=False 時に使用）────────────
LGB_NUM_BOOST_ROUND = 3000
LGB_EARLY_STOPPING  = 100

LGB_PARAMS_BINARY = {          # win / place3 / win_tie
    "objective": "binary",   "metric": "auc",
    "num_leaves": 63,        "learning_rate": 0.05,
    "colsample_bytree": 0.8, "subsample": 0.8,       "subsample_freq": 1,
    "reg_alpha": 0.1,        "reg_lambda": 0.1,       "min_child_samples": 20,
    "verbose": -1,           "random_state": 42,
}
LGB_PARAMS_REGRESSION = {      # speed_deviation
    "objective": "regression", "metric": "rmse",
    "num_leaves": 63,          "learning_rate": 0.05,
    "feature_fraction": 0.8,   "bagging_fraction": 0.8,  "bagging_freq": 1,
    "reg_alpha": 0.1,          "reg_lambda": 0.1,        "min_data_in_leaf": 20,
    "verbose": -1,             "random_state": 42,
}
LGB_PARAMS_RANK = {            # rank (LambdaRank)
    "objective": "lambdarank", "metric": "ndcg",
    "ndcg_eval_at": [1, 3, 5],
    "num_leaves": 63,          "learning_rate": 0.05,
    "feature_fraction": 0.8,   "bagging_fraction": 0.8,  "bagging_freq": 1,
    "reg_alpha": 0.1,          "reg_lambda": 0.1,        "min_data_in_leaf": 20,
    "verbose": -1,             "seed": 42,
}

# ── Optuna 設定 ───────────────────────────────────────────────────────────
USE_OPTUNA      = True   # True: Optuna 最適化 → 最終再学習 / False: デフォルト params で学習
OPTUNA_TRIALS   = 50      # トライアル数（目安: 30〜100）
OPTUNA_CV_FOLDS = 5       # Optuna 内 CV フォールド数
OPTUNA_TIMEOUT  = 600     # タイムアウト（秒）

# ── データ分割 ────────────────────────────────────────────────────────────
TEST_DAYS = 120           # 直近 N 日をテストセットに使用（時系列分割）

# ── 保存 ──────────────────────────────────────────────────────────────────
SAVE_MODEL = True         # True: 学習後に keiba/data/models/ へ保存

# ── 確認出力 ─────────────────────────────────────────────────────────────
_is_regression = TRAIN_TARGET in ("speed_deviation",)
_is_ranking    = TRAIN_TARGET == "rank"
print(f"{'='*55}")
print(f"TARGET       : {TRAIN_TARGET}  ({'回帰' if _is_regression else 'ランキング' if _is_ranking else '2値分類'})")
print(f"USE_OPTUNA   : {USE_OPTUNA}"
      + (f"  (trials={OPTUNA_TRIALS}, timeout={OPTUNA_TIMEOUT}s, cv={OPTUNA_CV_FOLDS})" if USE_OPTUNA else ""))
print(f"TEST_DAYS    : {TEST_DAYS}")
print(f"SAVE_MODEL   : {SAVE_MODEL}")
print(f"{'='*55}")
if TRAIN_TARGET != TARGET:
    print(f"\n⚠ Cell 7 の TARGET={TARGET!r} と異なります。Cell F でパイプラインを再実行します。")

In [ ]:
## ── Cell F: データ準備 + Train/Test 分割 ────────────────────────────────
from keiba_ai.train import _make_target

# ── 前処理済みフレームの選択（TRAIN_TARGET が Cell 7 と同じなら再利用）───
if TRAIN_TARGET != TARGET:
    print(f"Cell 7 の TARGET={TARGET!r} と異なるため、パイプラインを再実行します...")
    _df_pre_nb, _opt_nb, _cat_nb = prepare_for_lightgbm_ultimate(
        df.copy(), target_col=TRAIN_TARGET, is_training=True,
    )
    print(f"  再実行後: {_df_pre_nb.shape[1]} 列  cat: {len(_cat_nb)}")
else:
    _df_pre_nb, _cat_nb = df_pre, cat_features
    print(f"Cell 7 の前処理結果を再利用  ({_df_pre_nb.shape[1]} 列  cat: {len(_cat_nb)})")

# ── ターゲット生成 ────────────────────────────────────────────────────────
y_nb = _make_target(df, TRAIN_TARGET)

# speed_deviation の場合は NaN 行（時計欠損馬）を除外
if _is_regression:
    _valid_mask = y_nb.notna()
    y_nb       = y_nb[_valid_mask]
    _df_pre_nb = _df_pre_nb.loc[_valid_mask]
    print(f"NaN除外後: {_valid_mask.sum():,}行 (除外: {(~_valid_mask).sum():,}行)")

print(f"\nターゲット分布:")
print(y_nb.describe().to_string() if _is_regression else y_nb.value_counts().to_string())

# ── 特徴量列を確定 ────────────────────────────────────────────────────────
# 識別子・ターゲット列・object型列（LightGBM が受け付けない）を除外
_id_exclude = {"race_id", "horse_id", "horse_name", "jockey_name", TRAIN_TARGET,
               "finish", "time", "time_seconds", "speed_deviation", "rank",
               "win", "place3", "win_tie"}
_feature_cols_nb = [
    c for c in _df_pre_nb.columns
    if c not in _id_exclude and _df_pre_nb[c].dtype != object
]
_obj_dropped = [
    c for c in _df_pre_nb.columns
    if c not in _id_exclude and _df_pre_nb[c].dtype == object
]
if _obj_dropped:
    print(f"\nobject型として除外 ({len(_obj_dropped)}列): {_obj_dropped}")

X_nb = _df_pre_nb[_feature_cols_nb]

# インデックスを揃える
_common_idx = X_nb.index.intersection(y_nb.index)
X_nb = X_nb.loc[_common_idx].reset_index(drop=True)
y_nb = y_nb.loc[_common_idx].reset_index(drop=True)
_df_for_split = df.loc[_common_idx].reset_index(drop=True)

# ── 時系列 Train/Test 分割 ──────────────────────────────────────────────
_race_dates_nb = pd.to_datetime(
    _df_for_split["race_id"].astype(str).str[:8], format="%Y%m%d", errors="coerce"
)
_cutoff_nb = _race_dates_nb.max() - pd.Timedelta(days=TEST_DAYS)
_tr_mask_nb = _race_dates_nb <= _cutoff_nb
_te_mask_nb = ~_tr_mask_nb

X_train_nb = X_nb.loc[_tr_mask_nb].reset_index(drop=True)
X_test_nb  = X_nb.loc[_te_mask_nb].reset_index(drop=True)
y_train_nb = y_nb.loc[_tr_mask_nb].reset_index(drop=True)
y_test_nb  = y_nb.loc[_te_mask_nb].reset_index(drop=True)

# rank の場合: レースごとのグループサイズを計算
if _is_ranking:
    from collections import Counter as _Counter
    _race_ids_tr = _df_for_split.loc[_tr_mask_nb, "race_id"].values
    _race_ids_te = _df_for_split.loc[_te_mask_nb, "race_id"].values
    _group_train_nb = list(_Counter(_race_ids_tr).values())
    _group_test_nb  = list(_Counter(_race_ids_te).values())

# カテゴリカル特徴量のインデックス（object型除外後の列名ベースで再計算）
_cat_idx_nb = [
    X_train_nb.columns.get_loc(c)
    for c in _cat_nb
    if c in X_train_nb.columns
]

print(f"\n{'='*55}")
print(f"学習データ : {len(X_train_nb):,}行  (〜 {_cutoff_nb.date()})")
print(f"テストデータ: {len(X_test_nb):,}行  ({_cutoff_nb.date()} 〜)")
print(f"特徴量数   : {len(_feature_cols_nb)} 列  (カテゴリカル: {len(_cat_idx_nb)})")
print(f"{'='*55}")

In [ ]:
## ── Cell G: LightGBM 学習 / Optuna 最適化 ──────────────────────────────
import lightgbm as lgb
import time
from sklearn.metrics import roc_auc_score, log_loss
from scipy.stats import spearmanr as _spearmanr

# ── パラメータ選択（ターゲット種別で自動選択）────────────────────────────
if _is_ranking:
    _base_params = LGB_PARAMS_RANK.copy()
elif _is_regression:
    _base_params = LGB_PARAMS_REGRESSION.copy()
else:
    _base_params = LGB_PARAMS_BINARY.copy()

# ============================================================
# STEP 1: Optuna ハイパーパラメータ最適化（USE_OPTUNA=True 時）
# ============================================================
_final_params     = _base_params.copy()
_final_num_rounds = LGB_NUM_BOOST_ROUND
_optuna_best_score = None

if USE_OPTUNA and not _is_ranking and not _is_regression:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    from keiba_ai.optuna_optimizer import OptunaLightGBMOptimizer

    print(f"{'='*55}")
    print(f"Optuna 最適化開始  trials={OPTUNA_TRIALS}  cv={OPTUNA_CV_FOLDS}  timeout={OPTUNA_TIMEOUT}s")
    print(f"{'='*55}")

    _opt = OptunaLightGBMOptimizer(
        n_trials      = OPTUNA_TRIALS,
        cv_folds      = OPTUNA_CV_FOLDS,
        random_state  = 42,
        timeout       = OPTUNA_TIMEOUT,
    )
    _best_params_raw, _optuna_best_score = _opt.optimize(
        X_train_nb, y_train_nb,
        categorical_features=_cat_idx_nb,
    )
    _optimized = _opt.get_best_model_params()
    _final_num_rounds = int(_optimized.pop("n_estimators", LGB_NUM_BOOST_ROUND))
    _final_params.update(_optimized)
    print(f"\nOptuna ベストスコア: {_optuna_best_score:.4f}")
    print(f"最適ラウンド数     : {_final_num_rounds}")
    print("最適パラメータ:")
    for k, v in sorted(_optimized.items()):
        default_v = _base_params.get(k, "—")
        changed = " ← 変更" if str(v) != str(default_v) else ""
        print(f"  {k:30s}: {v}{changed}")
else:
    if USE_OPTUNA and _is_ranking:
        print("⚠ LambdaRank は Optuna 未対応 → デフォルトパラメータで学習します")
    print(f"デフォルトパラメータで学習  (USE_OPTUNA={USE_OPTUNA})")

# ============================================================
# STEP 2: 最終モデル学習
# ============================================================
print(f"\n{'='*55}")
print(f"LightGBM 学習開始  target={TRAIN_TARGET}  rounds={_final_num_rounds}")
print(f"{'='*55}")
_t0 = time.time()

_callbacks = [
    lgb.early_stopping(stopping_rounds=LGB_EARLY_STOPPING, verbose=False),
    lgb.log_evaluation(period=200),
]

if _is_ranking:
    _train_ds = lgb.Dataset(X_train_nb.values, label=y_train_nb.values, group=_group_train_nb)
    _valid_ds = lgb.Dataset(X_test_nb.values,  label=y_test_nb.values,  group=_group_test_nb)
    _final_params["label_gain"] = list(range(int(y_train_nb.max()) + 2))
else:
    _train_ds = lgb.Dataset(X_train_nb, y_train_nb, categorical_feature=_cat_idx_nb)
    _valid_ds = lgb.Dataset(X_test_nb,  y_test_nb,  reference=_train_ds)

model_nb = lgb.train(
    _final_params,
    _train_ds,
    num_boost_round = _final_num_rounds,
    valid_sets      = [_valid_ds],
    callbacks       = _callbacks,
)
_elapsed = time.time() - _t0

# ============================================================
# STEP 3: 評価
# ============================================================
_y_pred_nb = model_nb.predict(X_test_nb.values if _is_ranking else X_test_nb)

if _is_ranking:
    _sp, _ = _spearmanr(y_test_nb.values, _y_pred_nb)
    _score_label = f"Spearman = {_sp:.4f}"
    _metrics_nb  = {"spearman": float(_sp)}
elif _is_regression:
    _sp, _   = _spearmanr(y_test_nb.values, _y_pred_nb)
    _rmse_nb = float(np.sqrt(((y_test_nb.values - _y_pred_nb) ** 2).mean()))
    _score_label = f"Spearman = {_sp:.4f}  RMSE = {_rmse_nb:.4f}"
    _metrics_nb  = {"spearman": float(_sp), "rmse": _rmse_nb}
else:
    _auc_nb = roc_auc_score(y_test_nb.values, _y_pred_nb)
    _ll_nb  = log_loss(y_test_nb.values, _y_pred_nb)
    _score_label = f"AUC = {_auc_nb:.4f}  LogLoss = {_ll_nb:.4f}"
    _metrics_nb  = {"auc": float(_auc_nb), "logloss": float(_ll_nb)}

print(f"\n{'='*55}")
print(f"【学習完了】{_score_label}  ({_elapsed:.1f}s)")
print(f"{'='*55}")

# ── 特徴量重要度 TOP 20 ─────────────────────────────────────────────────
_imp_nb = pd.DataFrame({
    "feature":    _feature_cols_nb[:len(model_nb.feature_importance())],
    "importance": model_nb.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).reset_index(drop=True)

fig_g, ax_g = plt.subplots(figsize=(10, 6))
top20 = _imp_nb.head(20)
ax_g.barh(top20["feature"][::-1], top20["importance"][::-1], color="#4C78A8")
ax_g.set_xlabel("Importance (gain)")
ax_g.set_title(f"特徴量重要度 TOP 20  [{TRAIN_TARGET}]  {_score_label}")
plt.tight_layout()
plt.show()

In [ ]:
import inspect

from keiba_ai.optuna_optimizer import OptunaLightGBMOptimizer as _OptunaLightGBMOptimizer

if "is_regression" not in inspect.signature(_OptunaLightGBMOptimizer.__init__).parameters:
    _orig_optuna_init = _OptunaLightGBMOptimizer.__init__

    def _compat_optuna_init(self, *args, is_regression=None, **kwargs):
        return _orig_optuna_init(self, *args, **kwargs)

    _OptunaLightGBMOptimizer.__init__ = _compat_optuna_init

In [ ]:
## ── Cell H: モデル保存 + 評価サマリー ────────────────────────────────────
from datetime import datetime, timezone, timedelta
import joblib

_JST = timezone(timedelta(hours=9))
_ts  = datetime.now(tz=_JST).strftime("%Y%m%d_%H%M%S")
_save_dir = ROOT / "keiba" / "data" / "models"
_save_dir.mkdir(parents=True, exist_ok=True)
_save_path = _save_dir / f"model_{TRAIN_TARGET}_{_ts}.joblib"
_latest_path = _save_dir / "model_latest.joblib"

# ── 本番バンドル形式で保存（predict.py が直接ロードできる形式）──────────
_bundle_nb = {
    "model":              model_nb,                   # lgb.Booster
    "feature_cols_num":   _feature_cols_nb,           # 特徴量列名
    "feature_cols_cat":   _cat_nb,                    # カテゴリカル列名
    "target":             TRAIN_TARGET,
    "model_type":         "lightgbm_ranker" if _is_ranking else "lightgbm",
    "metrics":            _metrics_nb,
    "created_at":         _ts,
    "feature_importance": _imp_nb,
    "calibrator":         None,                       # 必要なら IsotonicRegression を追加
    "optuna_executed":    USE_OPTUNA and not _is_ranking,
    "optuna_best_score":  _optuna_best_score,
    "best_params":        _final_params,
    "num_boost_round":    model_nb.num_trees(),
    "date_range":         {"start": DATE_START, "end": DATE_END},
    "test_days":          TEST_DAYS,
}

if SAVE_MODEL:
    joblib.dump(_bundle_nb, _save_path)
    joblib.dump(_bundle_nb, _latest_path)   # model_latest も上書き
    print(f"保存完了: {_save_path.name}  ({_save_path.stat().st_size/1e6:.1f} MB)")
    print(f"         → {_latest_path} も更新")
else:
    print("SAVE_MODEL=False のためスキップ")

# ── 評価サマリーテーブル ──────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"【学習サマリー】")
print(f"{'='*65}")

_summary_rows = [
    ("期間",       f"{DATE_START} 〜 {DATE_END}"),
    ("TARGET",     TRAIN_TARGET),
    ("学習行数",   f"{len(X_train_nb):,}"),
    ("テスト行数", f"{len(X_test_nb):,}"),
    ("特徴量数",   f"{len(_feature_cols_nb)}"),
    ("ブースト回数", f"{model_nb.num_trees()} (max={_final_num_rounds})"),
    ("Optuna",     f"実施 (trials={OPTUNA_TRIALS})" if (USE_OPTUNA and not _is_ranking) else "なし"),
]
for m, v in _metrics_nb.items():
    _summary_rows.append((m.upper(), f"{v:.4f}"))
if _optuna_best_score is not None:
    _summary_rows.append(("Optuna CV スコア", f"{_optuna_best_score:.4f}"))

_sum_df = pd.DataFrame(_summary_rows, columns=["項目", "値"])
display(_sum_df.style.set_caption("学習結果サマリー").hide(axis="index"))

# ── TOP 20 重要度テーブルも表示 ──────────────────────────────────────────
print(f"\n【特徴量重要度 TOP 20 (gain)】")
display(_imp_nb.head(20).style
        .background_gradient(subset=["importance"], cmap="Blues")
        .set_caption(f"重要度 TOP 20 — {TRAIN_TARGET} ({_ts})"))

# 断捨離シミュレーション

断捨離レポート（セル 12）の結果を使って、**特徴量を削除したときに精度がどう変わるか**をノートブック上で確認します。

## 実行フロー

```
Cell I（削除設定）  →  Cell J（削除プレビュー / ドライラン）  →  Cell K（再学習 + 比較）
```

| セル | 内容 |
|---|---|
| **Cell I** | どのアクションカテゴリ・個別列を削除するか設定 |
| **Cell J** | 削除後の特徴量一覧・NaN率改善をプレビュー（再学習なし・高速）|
| **Cell K** | 削除後のデータで再学習し、Before/After を比較 |

In [ ]:
## ── Cell I: 断捨離シミュレーション設定 ────────────────────────────────

# ── ① 削除するアクションカテゴリ（report_df の「推奨アクション」列の値）──
# 下記から削除したいカテゴリをリストに入れる（部分一致）
PRUNE_ACTION_PREFIXES = [
    "削除推奨",          # lap_* / holding_* / training_* など高NaN列
    # "再スクレイプで改善",  # sf_* / holding_just_* など（未収集だが存在する）
    # "前処理: 最頻値補完 or 削除",  # corner_* / running_style_*
]

# ── ② 手動で追加削除する列名（report_df にない列でも可）──────────────────
PRUNE_EXTRA_COLS: list[str] = [
    # 例: "horse_weight_change",
]

# ── ③ カテゴリ削除対象でも強制的に残す列名 ─────────────────────────────
PRUNE_KEEP_COLS: list[str] = [
    # 例: "holding_just_speed",  # 保持したい列
]

# ─────────────────────────────────────────────────────────────────────────
# 削除対象列を report_df から自動収集
_prune_by_action = set()
for _prefix in PRUNE_ACTION_PREFIXES:
    _matched = report_df[report_df["推奨アクション"].str.startswith(_prefix)]["column"].tolist()
    _prune_by_action.update(_matched)

# 手動追加・手動除外を適用
_prune_cols_all = (
    (_prune_by_action | set(PRUNE_EXTRA_COLS)) - set(PRUNE_KEEP_COLS)
)

# 現在の学習特徴量（X_train_nb）のうち実際に存在するもののみ対象
_prune_in_model     = sorted(_prune_cols_all & set(_feature_cols_nb))
_prune_not_in_model = sorted(_prune_cols_all - set(_feature_cols_nb))
_pruned_feature_cols = [c for c in _feature_cols_nb if c not in _prune_cols_all]

# ─────────────────────────────────────────────────────────────────────────
print(f"{'='*60}")
print(f"【断捨離シミュレーション設定】")
print(f"{'='*60}")
print(f"削除アクション: {PRUNE_ACTION_PREFIXES}")
print()
print(f"  削除対象（モデル入力列）: {len(_prune_in_model):3d} 列")
print(f"  削除対象（すでに除外）  : {len(_prune_not_in_model):3d} 列  (参考)")
print(f"  手動追加削除           : {len(PRUNE_EXTRA_COLS):3d} 列")
print(f"  手動保持               : {len(PRUNE_KEEP_COLS):3d} 列")
print()
print(f"  削除前 特徴量数: {len(_feature_cols_nb)}")
print(f"  削除後 特徴量数: {len(_pruned_feature_cols)}")
print(f"  削減数        : {len(_feature_cols_nb) - len(_pruned_feature_cols)}")
print()
if _prune_in_model:
    print(f"【削除される列（モデル入力）】")
    for c in _prune_in_model:
        _nr = report_df.loc[report_df["column"] == c, "nan_%_before"].values
        _nr_str = f"NaN={_nr[0]:.0f}%" if len(_nr) else ""
        _act = report_df.loc[report_df["column"] == c, "推奨アクション"].values
        _act_str = _act[0].replace("\n", " ") if len(_act) else ""
        print(f"  {c:45s}  {_nr_str:9s}  {_act_str}")

In [ ]:
## ── Cell J: 断捨離プレビュー（ドライラン — 再学習なし）─────────────────

# 削除後の特徴量の NaN率・グループ分布を確認する（高速・数秒）

_pruned_X  = X_nb[_pruned_feature_cols]
_pruned_nan = _pruned_X.isna().mean()

# ── グループ別 残存列数を集計 ─────────────────────────────────────────────
_group_before, _group_after = {}, {}
for _grp, _gcols in FEATURE_GROUPS.items():
    _gname = _grp.split("(")[0][:25].strip()
    _before = [c for c in _gcols if c in _feature_cols_nb]
    _after  = [c for c in _gcols if c in _pruned_feature_cols]
    if _before:
        _group_before[_gname] = len(_before)
        _group_after[_gname]  = len(_after)

_groups   = list(_group_before.keys())
_b_counts = [_group_before[g] for g in _groups]
_a_counts = [_group_after[g]  for g in _groups]

fig_j, axes_j = plt.subplots(1, 3, figsize=(18, 6))

# 左: グループ別 削除前後の列数
x_j = np.arange(len(_groups))
w_j = 0.35
axes_j[0].barh(x_j + w_j/2, _b_counts, w_j, label="削除前", color="#4C78A8")
axes_j[0].barh(x_j - w_j/2, _a_counts, w_j, label="削除後", color="#54A24B")
axes_j[0].set_yticks(x_j)
axes_j[0].set_yticklabels(_groups, fontsize=8)
axes_j[0].set_xlabel("列数")
axes_j[0].set_title("グループ別 残存列数")
axes_j[0].legend()
for i, (b, a) in enumerate(zip(_b_counts, _a_counts)):
    axes_j[0].text(b + 0.1, i + w_j/2, str(b), va="center", fontsize=7, color="#4C78A8")
    axes_j[0].text(a + 0.1, i - w_j/2, str(a), va="center", fontsize=7, color="#54A24B")

# 中: 削除後の NaN率分布
_nan_bins  = [0, 0.01, 0.2, 0.5, 1.01]
_nan_labels = ["0%", "1-20%", "20-50%", ">50%"]
_before_counts_j = [
    ((_pruned_nan[_feature_cols_nb[i]] if _feature_cols_nb[i] in _pruned_nan.index else 0) == 0
     for i in range(len(_feature_cols_nb))),
]
_counts_before = [
    (pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]) == 0).sum(),
    ((pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]) > 0.01) &
     (pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]) <= 0.2)).sum(),
    ((pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]) > 0.2) &
     (pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]) <= 0.5)).sum(),
    (pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]) > 0.5).sum(),
]
_counts_after = [
    (_pruned_nan == 0).sum(),
    ((_pruned_nan > 0.01) & (_pruned_nan <= 0.2)).sum(),
    ((_pruned_nan > 0.2)  & (_pruned_nan <= 0.5)).sum(),
    (_pruned_nan > 0.5).sum(),
]
x_nan = np.arange(len(_nan_labels))
axes_j[1].bar(x_nan - 0.2, _counts_before, 0.35, label="削除前", color="#4C78A8")
axes_j[1].bar(x_nan + 0.2, _counts_after,  0.35, label="削除後", color="#54A24B")
axes_j[1].set_xticks(x_nan)
axes_j[1].set_xticklabels(_nan_labels)
axes_j[1].set_ylabel("列数")
axes_j[1].set_title("NaN率分布 削除前後比較")
axes_j[1].legend()
for i, (b, a) in enumerate(zip(_counts_before, _counts_after)):
    if b > 0: axes_j[1].text(i - 0.2, b + 0.3, str(b), ha="center", fontsize=8, color="#4C78A8")
    if a > 0: axes_j[1].text(i + 0.2, a + 0.3, str(a), ha="center", fontsize=8, color="#54A24B")

# 右: サマリーテキスト
axes_j[2].axis("off")
_summary_text = [
    ["項目",           "削除前",                   "削除後"],
    ["特徴量数",       str(len(_feature_cols_nb)), str(len(_pruned_feature_cols))],
    ["削減数",         "",                         f"▼ {len(_feature_cols_nb)-len(_pruned_feature_cols)}"],
    ["NaN=0% 列",      str(_counts_before[0]),     str(_counts_after[0])],
    ["NaN>50% 列",     str(_counts_before[3]),     str(_counts_after[3])],
    ["平均NaN率",
     f"{pd.Series([X_nb[c].isna().mean() for c in _feature_cols_nb]).mean()*100:.1f}%",
     f"{_pruned_nan.mean()*100:.1f}%"],
]
tbl_j = axes_j[2].table(
    cellText=_summary_text[1:],
    colLabels=_summary_text[0],
    cellLoc="center", loc="center",
    colWidths=[0.35, 0.3, 0.3],
)
tbl_j.auto_set_font_size(False)
tbl_j.set_fontsize(10)
tbl_j.scale(1, 1.8)
axes_j[2].set_title("削除前後サマリー")

plt.suptitle(
    f"断捨離ドライラン  削除: {len(_prune_in_model)} 列  残: {len(_pruned_feature_cols)} 列",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

print(f"✓ プレビュー完了。再学習して精度を比較するには Cell K を実行してください。")

In [ ]:
## ── Cell K: 断捨離後 再学習 + Before/After 比較 ─────────────────────────

# Cell G で学習したモデル（削除前）と同じ params で再学習し、精度を比較する

import lightgbm as lgb
import time
from sklearn.metrics import roc_auc_score, log_loss
from scipy.stats import spearmanr as _spearmanr

# ── 削除後のデータセットを作成 ───────────────────────────────────────────
_p_cat_idx = [
    _pruned_X.columns.get_loc(c)
    for c in _cat_nb
    if c in _pruned_feature_cols
]
X_train_p = X_train_nb[_pruned_feature_cols]
X_test_p  = X_test_nb[_pruned_feature_cols]

print(f"削除後 特徴量: {len(_pruned_feature_cols)} 列  (カテゴリカル: {len(_p_cat_idx)})")
print(f"削除前 特徴量: {len(_feature_cols_nb)} 列")
print(f"\n再学習中...")

_t_p = time.time()
_params_p = _final_params.copy()    # Cell G と同じ最終パラメータを使用

if _is_ranking:
    _train_ds_p = lgb.Dataset(X_train_p.values, label=y_train_nb.values, group=_group_train_nb)
    _valid_ds_p = lgb.Dataset(X_test_p.values,  label=y_test_nb.values,  group=_group_test_nb)
    _params_p["label_gain"] = list(range(int(y_train_nb.max()) + 2))
else:
    _train_ds_p = lgb.Dataset(X_train_p, y_train_nb, categorical_feature=_p_cat_idx)
    _valid_ds_p = lgb.Dataset(X_test_p,  y_test_nb,  reference=_train_ds_p)

model_pruned = lgb.train(
    _params_p,
    _train_ds_p,
    num_boost_round = _final_num_rounds,
    valid_sets      = [_valid_ds_p],
    callbacks       = [
        lgb.early_stopping(stopping_rounds=LGB_EARLY_STOPPING, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
_elapsed_p = time.time() - _t_p

# ── 評価 ─────────────────────────────────────────────────────────────────
_y_pred_p = model_pruned.predict(X_test_p.values if _is_ranking else X_test_p)

if _is_ranking:
    _sp_p, _ = _spearmanr(y_test_nb.values, _y_pred_p)
    _metrics_p = {"spearman": float(_sp_p)}
    _score_p   = f"Spearman = {_sp_p:.4f}"
elif _is_regression:
    _sp_p, _  = _spearmanr(y_test_nb.values, _y_pred_p)
    _rmse_p   = float(np.sqrt(((y_test_nb.values - _y_pred_p) ** 2).mean()))
    _metrics_p = {"spearman": float(_sp_p), "rmse": _rmse_p}
    _score_p   = f"Spearman = {_sp_p:.4f}  RMSE = {_rmse_p:.4f}"
else:
    _auc_p  = roc_auc_score(y_test_nb.values, _y_pred_p)
    _ll_p   = log_loss(y_test_nb.values, _y_pred_p)
    _metrics_p = {"auc": float(_auc_p), "logloss": float(_ll_p)}
    _score_p   = f"AUC = {_auc_p:.4f}  LogLoss = {_ll_p:.4f}"

print(f"\n【削除後】 {_score_p}  ({_elapsed_p:.1f}s)")
print(f"【削除前】 {_score_label}")

# ── Before/After 可視化 ───────────────────────────────────────────────────
fig_k, axes_k = plt.subplots(1, 3, figsize=(17, 5))

# 左: 指標比較棒グラフ
_metric_keys = list(_metrics_nb.keys())
_vals_before = [_metrics_nb[k] for k in _metric_keys]
_vals_after  = [_metrics_p.get(k, float("nan")) for k in _metric_keys]
_x_k = np.arange(len(_metric_keys))
_bars_b = axes_k[0].bar(_x_k - 0.2, _vals_before, 0.35, label="削除前", color="#4C78A8")
_bars_a = axes_k[0].bar(_x_k + 0.2, _vals_after,  0.35, label="削除後", color="#54A24B")
axes_k[0].set_xticks(_x_k)
axes_k[0].set_xticklabels([k.upper() for k in _metric_keys])
axes_k[0].set_title("評価指標 Before/After")
axes_k[0].legend()
for _b, _a, _k in zip(_vals_before, _vals_after, _metric_keys):
    _diff = _a - _b
    _color = "#54A24B" if (_k in ("spearman", "auc") and _diff >= 0) or \
                         (_k in ("rmse", "logloss") and _diff <= 0) else "#E45756"
    axes_k[0].text(_x_k[_metric_keys.index(_k)] + 0.2, _a + 0.002,
                   f"{_diff:+.4f}", ha="center", va="bottom", fontsize=9, color=_color)

# 中: 削除後モデルの特徴量重要度 TOP 20
_imp_p = pd.DataFrame({
    "feature":    _pruned_feature_cols[:len(model_pruned.feature_importance())],
    "importance": model_pruned.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(20)
axes_k[1].barh(_imp_p["feature"][::-1], _imp_p["importance"][::-1], color="#54A24B")
axes_k[1].set_xlabel("Importance (gain)")
axes_k[1].set_title(f"重要度 TOP 20 — 削除後 ({len(_pruned_feature_cols)} 列)")

# 右: サマリーテーブル
axes_k[2].axis("off")
_cmp_rows = [["項目", "削除前", "削除後", "差分"]]
_cmp_rows.append(["特徴量数",
                  str(len(_feature_cols_nb)),
                  str(len(_pruned_feature_cols)),
                  f"▼{len(_feature_cols_nb)-len(_pruned_feature_cols)}"])
_cmp_rows.append(["ブースト回数",
                  str(model_nb.num_trees()),
                  str(model_pruned.num_trees()),
                  f"{model_pruned.num_trees()-model_nb.num_trees():+d}"])
for _k in _metric_keys:
    _b_v = _metrics_nb.get(_k, float("nan"))
    _a_v = _metrics_p.get(_k, float("nan"))
    _d_v = _a_v - _b_v
    _cmp_rows.append([
        _k.upper(),
        f"{_b_v:.4f}",
        f"{_a_v:.4f}",
        f"{_d_v:+.4f}",
    ])

_tbl_k = axes_k[2].table(
    cellText=_cmp_rows[1:],
    colLabels=_cmp_rows[0],
    cellLoc="center", loc="center",
    colWidths=[0.35, 0.22, 0.22, 0.22],
)
_tbl_k.auto_set_font_size(False)
_tbl_k.set_fontsize(10)
_tbl_k.scale(1, 1.9)
# 改善した行をハイライト
for _r_idx, _k in enumerate(_metric_keys, start=1):
    _diff = _metrics_p.get(_k, 0) - _metrics_nb.get(_k, 0)
    _improved = (_k in ("spearman", "auc") and _diff > 0) or \
                (_k in ("rmse", "logloss") and _diff < 0)
    _color = "#1a3a1a" if _improved else "#3a1a1a"
    for _col_idx in range(4):
        _tbl_k[_r_idx + 2, _col_idx].set_facecolor(_color)
axes_k[2].set_title("比較サマリー")

plt.suptitle(
    f"断捨離シミュレーション  {TRAIN_TARGET}  削除 {len(_prune_in_model)} 列 → 残 {len(_pruned_feature_cols)} 列",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

# ── テキストサマリー ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"【断捨離シミュレーション結果】")
for _k in _metric_keys:
    _b_v = _metrics_nb.get(_k, float("nan"))
    _a_v = _metrics_p.get(_k, float("nan"))
    _d_v = _a_v - _b_v
    _up = "↑改善" if (_k in ("spearman","auc") and _d_v > 0) or \
                     (_k in ("rmse","logloss") and _d_v < 0) else "↓悪化"
    print(f"  {_k.upper():10s}: {_b_v:.4f} → {_a_v:.4f}  ({_d_v:+.4f}) {_up}")
print(f"\n  特徴量数: {len(_feature_cols_nb)} → {len(_pruned_feature_cols)} (-{len(_feature_cols_nb)-len(_pruned_feature_cols)}列)")

---
## 特徴量分析レポート — 速度偏差 (speed_deviation)

競馬 AI 実務運用を前提に、全 7 ステップで特徴量を評価する。

| ステップ | セル | 内容 |
|---|---|---|
| ① データ品質分析 | Cell L1 | 欠損率・ユニーク数・異常値・リーク可能性 |
| ② 単変量分析    | Cell L2 | Spearman 相関 / Mutual Information |
| ③ 多重共線性    | Cell L3 | Pearson 相関行列 / VIF |
| ④ モデルベース  | Cell L4 | Gain / Permutation / SHAP 重要度 |
| ⑤ グループ分析  | Cell L5 | 10 カテゴリ別統計 |
| ⑥ アブレーション| Cell L5 | グループ削除の影響推定（Permutation ベース） |
| ⑦ 最終評価     | Cell L6 | S/A/B/C ランク + 必須 / 推奨 / 削除候補 |

In [ ]:
## ── Cell L1: ① データ品質分析 ──────────────────────────────────────────
# 欠損率・ユニーク数・異常値 (3×IQR法)・リーク可能性 を評価する

import warnings

# ── NaN 率 ────────────────────────────────────────────────────────────────
_l1_nan = X_nb[_feature_cols_nb].isna().mean().rename("nan_rate")

# ── ユニーク数 ────────────────────────────────────────────────────────────
_l1_uniq = X_nb[_feature_cols_nb].nunique().rename("nunique")

# ── 外れ値率 (3×IQR, サンプル 50K 行で高速化) ───────────────────────────
_n_q1 = min(50_000, len(X_nb))
_Xq   = X_nb[_feature_cols_nb].sample(n=_n_q1, random_state=42)

def _iqr_outlier_rate(s):
    s = s.dropna()
    if len(s) < 10: return 0.0
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:     return 0.0
    return float(((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).mean())

_l1_out = _Xq.apply(_iqr_outlier_rate).rename("outlier_rate")

# ── リーク可能性チェック ─────────────────────────────────────────────────
# FUTURE_FIELDS との一致 = HIGH  / 着順・タイム系ワード含有 = MEDIUM
_leak_kw = ("_finish", "_time_", "speed_deviation", "_rank", "payout", "return")

def _leak_check(col: str) -> str:
    if col in FUTURE_FIELDS:             return "HIGH"
    if any(k in col for k in _leak_kw): return "MEDIUM"
    return "LOW"

_l1_leak = pd.Series({c: _leak_check(c) for c in _feature_cols_nb}, name="leak_risk")

# ── quality_df 構築 ──────────────────────────────────────────────────────
quality_df = pd.DataFrame({
    "nan_rate":     _l1_nan,
    "nunique":      _l1_uniq,
    "outlier_rate": _l1_out,
    "leak_risk":    _l1_leak,
}).reset_index().rename(columns={"index": "feature"})

_action_map = report_df.set_index("column")["推奨アクション"].to_dict()
quality_df["断捨離アクション"] = quality_df["feature"].map(_action_map).fillna("保持")

# ── 可視化 ────────────────────────────────────────────────────────────────
fig_l1, axes_l1 = plt.subplots(1, 3, figsize=(19, 8))

# 左: NaN率 TOP 30
_q_sorted = quality_df.sort_values("nan_rate", ascending=False).head(30)
_c_nan = ["#E45756" if r > 0.5 else "#F58518" if r > 0.2 else "#4C78A8"
          for r in _q_sorted["nan_rate"]]
axes_l1[0].barh(_q_sorted["feature"][::-1], _q_sorted["nan_rate"][::-1] * 100,
                color=_c_nan[::-1])
axes_l1[0].axvline(50, color="red",    ls="--", alpha=0.5, label=">50%")
axes_l1[0].axvline(20, color="orange", ls="--", alpha=0.5, label=">20%")
axes_l1[0].set_xlabel("NaN 率 (%)")
axes_l1[0].set_title("NaN 率 TOP 30")
axes_l1[0].legend(fontsize=8)

# 中: NaN率分布バー
_bins_l1  = [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.01]
_blabels  = ["0%", "〜1%", "〜5%", "〜10%", "〜20%", "〜50%", ">50%"]
_bcols    = ["#54A24B","#54A24B","#EECA3B","#EECA3B","#F58518","#E45756"]
_bhist    = [
    ((quality_df["nan_rate"] >= _bins_l1[i]) & (quality_df["nan_rate"] < _bins_l1[i+1])).sum()
    for i in range(len(_bins_l1)-1)
]
axes_l1[1].bar(_blabels[1:], _bhist, color=_bcols)
for i, v in enumerate(_bhist):
    if v > 0: axes_l1[1].text(i, v + 0.1, str(v), ha="center", fontsize=9)
axes_l1[1].set_ylabel("特徴量数")
axes_l1[1].set_title("NaN 率分布")

# 右: 品質サマリーテーブル
axes_l1[2].axis("off")
_lk = quality_df["leak_risk"].value_counts()
_tbl_data_l1 = [
    ["総特徴量数",          str(len(quality_df))],
    ["NaN = 0%",           str((quality_df["nan_rate"] == 0).sum())],
    ["NaN 1〜20%",         str(((quality_df["nan_rate"]>0.01)&(quality_df["nan_rate"]<=0.2)).sum())],
    ["NaN 20〜50%",        str(((quality_df["nan_rate"]>0.2 )&(quality_df["nan_rate"]<=0.5)).sum())],
    ["NaN > 50%",          str((quality_df["nan_rate"]>0.5).sum())],
    ["外れ値率 > 1%",      str((quality_df["outlier_rate"]>0.01).sum())],
    ["リーク HIGH",         str(_lk.get("HIGH", 0))],
    ["リーク MEDIUM",       str(_lk.get("MEDIUM", 0))],
    ["リーク LOW (安全)",   str(_lk.get("LOW", 0))],
]
_tbl_l1 = axes_l1[2].table(cellText=_tbl_data_l1, colLabels=["項目","件数"],
                             cellLoc="center", loc="center", colWidths=[0.6, 0.3])
_tbl_l1.auto_set_font_size(False); _tbl_l1.set_fontsize(10); _tbl_l1.scale(1, 1.8)
axes_l1[2].set_title("データ品質サマリー")

plt.suptitle("① データ品質分析  (NaN率・外れ値・リーク可能性)", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

# ── テキスト出力 ─────────────────────────────────────────────────────────
print("【NaN率 > 20% 特徴量】")
for _, r in quality_df[quality_df["nan_rate"]>0.2].sort_values("nan_rate",ascending=False).iterrows():
    print(f"  {r['feature']:45s}  NaN={r['nan_rate']*100:.1f}%  "
          f"{r['断捨離アクション'].replace(chr(10),' ')}")
print(f"\n【リーク MEDIUM/HIGH 特徴量】")
for _, r in quality_df[quality_df["leak_risk"]!="LOW"].iterrows():
    print(f"  {r['feature']:45s}  {r['leak_risk']}")
print(f"\n✓ quality_df 構築完了 ({len(quality_df)} 列)")

In [ ]:
## ── Cell L2: ② 単変量分析 (Spearman 相関 + Mutual Information) ─────────
from scipy.stats import spearmanr as _scipy_spearmanr
from sklearn.feature_selection import mutual_info_regression

# サンプリング (MI は計算コスト大 → 20K 行)
_n_l2    = min(20_000, len(X_nb))
_l2_idx  = X_nb.sample(n=_n_l2, random_state=42).index
_Xl2     = X_nb.loc[_l2_idx, _feature_cols_nb].copy()
_yl2     = y_nb.loc[_l2_idx].values

# ── Spearman 相関 ─────────────────────────────────────────────────────────
print(f"Spearman 計算中 ({len(_feature_cols_nb)} 列 × {_n_l2:,} 行)...")
_sp_dict = {}
for c in _feature_cols_nb:
    _col = _Xl2[c].fillna(_Xl2[c].median())
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _r, _ = _scipy_spearmanr(_col.values, _yl2)
    _sp_dict[c] = float(_r) if not np.isnan(_r) else 0.0
_sp_s = pd.Series(_sp_dict, name="spearman")

# ── Mutual Information ────────────────────────────────────────────────────
print("Mutual Information 計算中...")
_Xl2_filled = _Xl2.fillna(_Xl2.median())
_mi_arr = mutual_info_regression(_Xl2_filled.values, _yl2, random_state=42, n_neighbors=5)
_mi_s   = pd.Series(dict(zip(_feature_cols_nb, _mi_arr)), name="MI")

# ── univariate_df ────────────────────────────────────────────────────────
univariate_df = pd.DataFrame({
    "spearman":     _sp_s,
    "spearman_abs": _sp_s.abs(),
    "MI":           _mi_s,
    "MI_norm":      _mi_s / _mi_s.max() if _mi_s.max() > 0 else _mi_s,
}).reset_index().rename(columns={"index": "feature"})
univariate_df = univariate_df.sort_values("spearman_abs", ascending=False).reset_index(drop=True)

# ── 可視化 ────────────────────────────────────────────────────────────────
fig_l2, axes_l2 = plt.subplots(1, 2, figsize=(18, 11))

_top_n_l2 = 35
_sp_top   = univariate_df.head(_top_n_l2)
_c_sp     = ["#E45756" if v < 0 else "#4C78A8" for v in _sp_top["spearman"]]
axes_l2[0].barh(_sp_top["feature"][::-1], _sp_top["spearman"][::-1], color=_c_sp[::-1])
axes_l2[0].axvline(0, color="black", lw=0.5)
axes_l2[0].axvline( 0.1, color="green",  ls="--", alpha=0.4, label="|r|=0.1")
axes_l2[0].axvline(-0.1, color="green",  ls="--", alpha=0.4)
axes_l2[0].set_xlabel("Spearman 相関係数  (青=正, 赤=負)")
axes_l2[0].set_title(f"② Spearman 相関  |r| TOP {_top_n_l2}")
axes_l2[0].legend(fontsize=8)

_mi_top = univariate_df.nlargest(_top_n_l2, "MI")
axes_l2[1].barh(_mi_top["feature"][::-1], _mi_top["MI"][::-1], color="#54A24B")
axes_l2[1].set_xlabel("Mutual Information")
axes_l2[1].set_title(f"② Mutual Information  TOP {_top_n_l2}")

plt.suptitle("② 単変量分析  (Spearman + Mutual Information  ←  速度偏差)", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# ── 統計サマリー ─────────────────────────────────────────────────────────
print(f"\n【Spearman |r| 分布】")
for _thr in (0.15, 0.10, 0.05, 0.02, 0.01):
    _cnt = (univariate_df["spearman_abs"] >= _thr).sum()
    print(f"  |r| ≥ {_thr:.2f}: {_cnt:3d} 列")
print(f"  |r| < 0.01 (ほぼ無相関): {(univariate_df['spearman_abs'] < 0.01).sum()} 列")

print(f"\n✓ univariate_df 構築完了 ({len(univariate_df)} 列)")

In [ ]:
## ── Cell L3: ③ 多重共線性分析 (Pearson 相関行列 + VIF) ─────────────────
# 高相関ペアを特定し、VIF が大きい列（多重共線性が強い列）を洗い出す

# ── Pearson 相関行列 (サンプル 30K) ───────────────────────────────────────
print("Pearson 相関行列を計算中...")
_n_l3 = min(30_000, len(X_nb))
_Xl3  = X_nb[_feature_cols_nb].sample(n=_n_l3, random_state=42).fillna(0)
_corr = _Xl3.corr(method="pearson")

# 高相関ペア (|r| ≥ 0.85) を抽出
_high_pairs = []
_fc = _feature_cols_nb
for _i in range(len(_fc)):
    for _j in range(_i + 1, len(_fc)):
        _rv = _corr.iloc[_i, _j]
        if abs(_rv) >= 0.85:
            _high_pairs.append({"A": _fc[_i], "B": _fc[_j], "r": round(_rv, 4)})

high_corr_df = (pd.DataFrame(_high_pairs).sort_values("r", key=abs, ascending=False)
                if _high_pairs else pd.DataFrame(columns=["A","B","r"]))
print(f"高相関ペア (|r| ≥ 0.85): {len(high_corr_df)} ペア")

# ── VIF ─────────────────────────────────────────────────────────────────
# statsmodels がなければ手動計算 (相関行列の逆数から近似)
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor as _vif_fn
    _n_vif  = min(8_000, len(X_nb))
    _Xvif   = X_nb[_feature_cols_nb].sample(n=_n_vif, random_state=42).fillna(0)
    _vif_cols = [c for c in _feature_cols_nb if _Xvif[c].std() > 1e-6]
    _Xvif_mat = _Xvif[_vif_cols].values
    print(f"VIF 計算中 ({len(_vif_cols)} 列 × {_n_vif} 行)...")
    _vif_vals = []
    for _k in range(len(_vif_cols)):
        try:    _vif_vals.append(min(float(_vif_fn(_Xvif_mat, _k)), 999.0))
        except: _vif_vals.append(float("nan"))
    vif_df = pd.DataFrame({"feature": _vif_cols, "VIF": _vif_vals}).sort_values("VIF", ascending=False).reset_index(drop=True)
    _vif_ok = True
    print(f"  VIF > 10: {(vif_df['VIF'] > 10).sum()}  VIF > 5: {(vif_df['VIF'] > 5).sum()}")
except ImportError:
    print("statsmodels 未インストール → 相関行列逆数から VIF を近似")
    _corr_np = np.array(_corr.values, dtype=float)
    np.fill_diagonal(_corr_np, 1.0)
    try:
        _inv = np.linalg.inv(_corr_np)
        _vif_approx = np.diag(_inv)
    except np.linalg.LinAlgError:
        _vif_approx = np.full(len(_fc), float("nan"))
    vif_df = pd.DataFrame({"feature": _fc, "VIF": [min(v,999.0) for v in _vif_approx]}).sort_values("VIF", ascending=False).reset_index(drop=True)
    _vif_ok = True

# ── ヒートマップ用列: 高相関ペアに登場した列 ─────────────────────────────
_hm_cols = list(dict.fromkeys(
    [r["A"] for _, r in high_corr_df.iterrows()] +
    [r["B"] for _, r in high_corr_df.iterrows()]
))[:40]
if not _hm_cols:
    _hm_cols = _feature_cols_nb[:25]

# ── 可視化 ────────────────────────────────────────────────────────────────
fig_l3, axes_l3 = plt.subplots(1, 2, figsize=(18, 9))

# 左: ヒートマップ
_c_sub = _corr.loc[_hm_cols, _hm_cols]
_im = axes_l3[0].imshow(_c_sub.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axes_l3[0].set_xticks(range(len(_hm_cols)))
axes_l3[0].set_yticks(range(len(_hm_cols)))
axes_l3[0].set_xticklabels(_hm_cols, rotation=90, fontsize=7)
axes_l3[0].set_yticklabels(_hm_cols, fontsize=7)
plt.colorbar(_im, ax=axes_l3[0], fraction=0.046, pad=0.04)
axes_l3[0].set_title(f"Pearson 相関ヒートマップ\n(高相関ペア登場列 {len(_hm_cols)} 列)")

# 右: VIF TOP 30
_vif_top = vif_df.dropna().head(30)
_c_vif   = ["#E45756" if v > 10 else "#F58518" if v > 5 else "#54A24B"
            for v in _vif_top["VIF"]]
axes_l3[1].barh(_vif_top["feature"][::-1], _vif_top["VIF"][::-1], color=_c_vif[::-1])
axes_l3[1].axvline(10, color="red",    ls="--", alpha=0.5, label="VIF=10 (強)")
axes_l3[1].axvline(5,  color="orange", ls="--", alpha=0.5, label="VIF=5  (中)")
axes_l3[1].set_xlabel("VIF")
axes_l3[1].set_title("VIF TOP 30  (赤>10, 橙>5)")
axes_l3[1].legend(fontsize=8)

plt.suptitle("③ 多重共線性分析  (Pearson 相関 + VIF)", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# テキスト出力
if len(high_corr_df) > 0:
    print("\n【高相関ペア (|r| ≥ 0.85) TOP 20】")
    print(high_corr_df.head(20).to_string(index=False))
print(f"\n✓ vif_df / high_corr_df 構築完了")

In [ ]:
## ── Cell L4: ④ モデルベース分析 (Gain / Permutation / SHAP) ────────────
from scipy.stats import spearmanr as _scipy_spearmanr

# ── Gain Importance (既存モデルから即時取得) ─────────────────────────────
_l4_feat  = model_nb.feature_name()
_l4_gain  = model_nb.feature_importance("gain")
_l4_split = model_nb.feature_importance("split")
_gain_df = pd.DataFrame({
    "feature":    _l4_feat,
    "gain":       _l4_gain,
    "split":      _l4_split,
    "gain_norm":  _l4_gain / _l4_gain.sum() if _l4_gain.sum() > 0 else _l4_gain,
}).sort_values("gain", ascending=False).reset_index(drop=True)

print("【Gain TOP 10】")
for _, r in _gain_df.head(10).iterrows():
    print(f"  {r['feature']:45s}  gain={r['gain']:10,.0f}  split={r['split']:5.0f}")

# ── Permutation Importance ────────────────────────────────────────────────
print("\nPermutation Importance 計算中 (テストセット 3K サンプル)...")
_n_perm   = min(3_000, len(X_test_nb))
_Xp       = X_test_nb.iloc[:_n_perm]
_yp       = y_test_nb.iloc[:_n_perm].values
_y_base   = model_nb.predict(_Xp)
_sp_base, _ = _scipy_spearmanr(_yp, _y_base)

_perm_imp = {}
for c in _feature_cols_nb:
    _Xs = _Xp.copy()
    _Xs[c] = _Xs[c].sample(frac=1, random_state=42).values
    _ys, _ = _scipy_spearmanr(_yp, model_nb.predict(_Xs))
    _perm_imp[c] = float(_sp_base - _ys)

perm_df = (pd.DataFrame({"feature": list(_perm_imp.keys()),
                          "perm_imp": list(_perm_imp.values())})
           .sort_values("perm_imp", ascending=False).reset_index(drop=True))
print(f"  ベース Spearman={_sp_base:.4f}  |Perm TOP5|: {perm_df.head(5)['feature'].tolist()}")

# ── SHAP ─────────────────────────────────────────────────────────────────
_shap_ok = False
try:
    import shap as _shap
    print("\nSHAP 計算中 (2K サンプル)...")
    _n_shap = min(2_000, len(X_test_nb))
    _explainer = _shap.TreeExplainer(model_nb)
    _shap_vals = _explainer.shap_values(X_test_nb.iloc[:_n_shap], check_additivity=False)
    _shap_mean = np.abs(_shap_vals).mean(axis=0)
    shap_df = pd.DataFrame({"feature": _feature_cols_nb, "shap": _shap_mean}
                           ).sort_values("shap", ascending=False).reset_index(drop=True)
    _shap_ok = True
    print(f"  SHAP TOP5: {shap_df.head(5)['feature'].tolist()}")
except Exception as _shap_err:
    print(f"  SHAP スキップ ({type(_shap_err).__name__}: {_shap_err})")
    shap_df = pd.DataFrame({"feature": _feature_cols_nb, "shap": float("nan")})

# ── importance_df 統合 ────────────────────────────────────────────────────
importance_df = (
    _gain_df[["feature", "gain", "split", "gain_norm"]]
    .merge(perm_df[["feature", "perm_imp"]], on="feature", how="left")
    .merge(shap_df[["feature", "shap"]],    on="feature", how="left")
)

def _scale01(s):
    mn = s.min();  mx = s.max()
    return (s - mn) / (mx - mn + 1e-12) if mx > mn else s * 0

importance_df["gain_s"] = _scale01(importance_df["gain_norm"])
importance_df["perm_s"] = _scale01(importance_df["perm_imp"].clip(lower=0))
importance_df["shap_s"] = _scale01(importance_df["shap"])

_score_cols = (["gain_s", "perm_s", "shap_s"] if _shap_ok else ["gain_s", "perm_s"])
importance_df["composite"] = importance_df[_score_cols].mean(axis=1)
importance_df = importance_df.sort_values("composite", ascending=False).reset_index(drop=True)

# ── 可視化 ────────────────────────────────────────────────────────────────
_nc_l4  = 3 if _shap_ok else 2
fig_l4, axes_l4 = plt.subplots(1, _nc_l4, figsize=(18, 11))
_top_l4 = 30

_g30 = importance_df.head(_top_l4)
axes_l4[0].barh(_g30["feature"][::-1], _g30["gain"][::-1], color="#4C78A8")
axes_l4[0].set_title(f"Gain Importance TOP {_top_l4}"); axes_l4[0].set_xlabel("Gain")

_p30 = perm_df.head(_top_l4)
_cp  = ["#E45756" if v < 0 else "#54A24B" for v in _p30["perm_imp"]]
axes_l4[1].barh(_p30["feature"][::-1], _p30["perm_imp"][::-1], color=_cp[::-1])
axes_l4[1].axvline(0, color="black", lw=0.5)
axes_l4[1].set_title(f"Permutation Importance TOP {_top_l4}\n(Spearman 低下量: 大きい=重要)")
axes_l4[1].set_xlabel("ΔSpearman (削除時低下量)")

if _shap_ok:
    _s30 = shap_df.head(_top_l4)
    axes_l4[2].barh(_s30["feature"][::-1], _s30["shap"][::-1], color="#F58518")
    axes_l4[2].set_title(f"SHAP Mean |value| TOP {_top_l4}"); axes_l4[2].set_xlabel("Mean |SHAP|")

plt.suptitle("④ モデルベース重要度分析  (Gain / Permutation / SHAP)", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

print("\n【Composite Score TOP 20】")
print(importance_df[["feature","gain","perm_imp","composite"]].head(20).to_string(index=False))
print(f"\n✓ importance_df 構築完了 ({len(importance_df)} 列)")

In [ ]:
## ── Cell L5: ⑤ グループ分析 + ⑥ アブレーション推定 ─────────────────────
# 10 カテゴリに分類し、各グループの重要度・NaN率・削除影響を推定する

# ── 10 カテゴリ定義 ─────────────────────────────────────────────────────
_GMAP = {
    "①馬能力": [
        "prev_speed_index","prev_speed_zscore","prev2_speed_index","prev2_speed_zscore",
        "speed_index_change","speed_avg_weighted","speed_best_2","speed_improving",
        "recent_form_weighted","form_trend","recent_form_5race","form_consistency",
        "win_count_5","top3_count_5","speed_best_5","speed_trend_5",
        "horse_win_rate","horse_win_rate_exp","horse_show_rate_exp",
        "horse_speed_exp_mean","horse_speed_exp_std","horse_recent30_win_rate",
        "sf_index_last","sf_index_2ago","sf_index_3ago",
        "sf_max_index","sf_course_max_index","sf_dist_max_index","sf_index_trend",
        "race_avg_prev_speed","race_max_prev_speed","speed_vs_race_avg","horse_speed_rank_pct",
        "race_avg_prev_finish",
    ],
    "②騎手能力": [
        "jockey_win_rate","jockey_show_rate","jockey_course_win_rate",
    ],
    "③調教師能力": [
        "trainer_win_rate","trainer_show_rate",
    ],
    "④血統": [
        "sire_win_rate","sire_show_rate","sire_surface_win_rate","sire_dist_band_win_rate",
    ],
    "⑤距離適性": [
        "horse_distance_win_rate","horse_distance_avg_finish",
        "distance_change","distance_increased","distance_decreased",
    ],
    "⑥馬場適性": [
        "horse_surface_win_rate","horse_surface_avg_finish",
        "straight_length","track_type","corner_radius","inner_bias","inner_advantage",
    ],
    "⑦枠順": [
        "gate_bracket_win_rate","frame_race_type",
    ],
    "⑧人気": [
        "popularity_normalized","odds_rank_in_race",
    ],
    "⑨オッズ": [
        "implied_prob","log_odds","implied_prob_norm",
        "odds_z_in_race","market_entropy","top3_probability","odds_is_missing",
    ],
    "⑩クラス情報": [
        "race_class_num","class_rank_adj","weight_vs_standard",
        "prev_race_class_num","class_change","is_class_up","is_class_down",
        "is_same_class","class_diff_abs","horse_avg_class_num","class_drop",
    ],
}

# 残り → その他
_mapped_all = {c for cs in _GMAP.values() for c in cs}
_GMAP["⑪その他"] = [c for c in _feature_cols_nb if c not in _mapped_all]

# ── グループ統計テーブル ─────────────────────────────────────────────────
_gstats = []
for _gn, _gcols in _GMAP.items():
    _in = [c for c in _gcols if c in _feature_cols_nb]
    if not _in: continue
    _nan_mean  = quality_df.set_index("feature")["nan_rate"].reindex(_in).mean()
    _sp_mean   = univariate_df.set_index("feature")["spearman_abs"].reindex(_in).mean()
    _gain_sum  = importance_df.set_index("feature")["gain"].reindex(_in).sum()
    _perm_sum  = importance_df.set_index("feature")["perm_imp"].reindex(_in).sum()
    _comp_mean = importance_df.set_index("feature")["composite"].reindex(_in).mean()
    _gstats.append({
        "グループ":      _gn,
        "列数":          len(_in),
        "平均NaN率":     round(_nan_mean, 3),
        "平均|Spearman|":round(_sp_mean, 4),
        "Gain合計":      int(_gain_sum),
        "Perm合計ΔSp":   round(_perm_sum, 4),
        "平均Composite": round(_comp_mean, 4),
    })

group_df = pd.DataFrame(_gstats).sort_values("Gain合計", ascending=False).reset_index(drop=True)

print("【⑤ グループ別 特徴量統計】")
display(group_df.style
        .background_gradient(subset=["平均NaN率"],     cmap="Oranges")
        .background_gradient(subset=["Gain合計"],      cmap="Blues")
        .background_gradient(subset=["Perm合計ΔSp"],  cmap="Greens")
        .format({"平均NaN率": "{:.1%}", "Perm合計ΔSp": "{:+.4f}", "Gain合計": "{:,}"})
        .set_caption("⑤ グループ別 特徴量統計"))

# ── ⑥ アブレーション推定 ─────────────────────────────────────────────────
print(f"\n【⑥ アブレーション推定 (ベース Spearman = {_sp_base:.4f})】")
ablation_df = group_df[["グループ","列数","Perm合計ΔSp"]].copy()
ablation_df["削除後Sp推定"]  = (_sp_base - ablation_df["Perm合計ΔSp"]).round(4)
ablation_df["影響度"] = ablation_df["Perm合計ΔSp"].apply(
    lambda v: "★★★ 高" if v > 0.015 else "★★  中" if v > 0.005 else "★   低"
)
display(ablation_df.sort_values("Perm合計ΔSp", ascending=False).style
        .background_gradient(subset=["Perm合計ΔSp"], cmap="RdYlGn")
        .format({"Perm合計ΔSp": "{:+.4f}", "削除後Sp推定": "{:.4f}"})
        .set_caption("⑥ アブレーション分析（グループ削除の Spearman 低下推定）"))

# ── 可視化 ────────────────────────────────────────────────────────────────
fig_l5, axes_l5 = plt.subplots(1, 2, figsize=(16, 7))

# 左: アブレーション影響
_abl_s = ablation_df.sort_values("Perm合計ΔSp", ascending=True)
_c_abl = ["#E45756" if v > 0.015 else "#F58518" if v > 0.005 else "#54A24B"
          for v in _abl_s["Perm合計ΔSp"]]
axes_l5[0].barh(_abl_s["グループ"], _abl_s["Perm合計ΔSp"], color=_c_abl)
axes_l5[0].axvline(0, color="black", lw=0.5)
axes_l5[0].set_xlabel("ΔSpearman (削除時の低下推定値)")
axes_l5[0].set_title("⑥ グループ削除の影響推定\n(赤=高影響 / 緑=影響小)")

# 右: グループ別 Gain パイチャート
_gg = group_df[group_df["Gain合計"] > 0]
axes_l5[1].pie(_gg["Gain合計"], labels=_gg["グループ"].str[:8],
               autopct="%1.1f%%", startangle=140, textprops={"fontsize": 8})
axes_l5[1].set_title("グループ別 Gain Importance 割合")

plt.suptitle("⑤⑥ グループ分析 + アブレーション推定", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
print(f"\n✓ group_df / ablation_df 構築完了")

In [ ]:
## ── Cell L6: ⑦ 最終評価 — S/A/B/C ランク + 必須 / 推奨 / 削除候補 ────────
# 全分析スコアを統合し、競馬 AI 実務運用観点でランク付けする

from matplotlib.patches import Patch

# ── 各スコアを統合 ────────────────────────────────────────────────────────
def _s01(s): return (s - s.min()) / (s.max() - s.min() + 1e-12)

_final = importance_df[["feature", "composite", "gain", "perm_imp"]].copy()
_final = _final.merge(univariate_df[["feature","spearman","spearman_abs","MI_norm"]], on="feature", how="left")
_final = _final.merge(quality_df[["feature","nan_rate","leak_risk","断捨離アクション"]], on="feature", how="left")
_final = _final.merge(vif_df[["feature","VIF"]], on="feature", how="left")

# スコア成分 (0〜1)
_final["s_model"]  = _s01(_final["composite"])
_final["s_univ"]   = (_final["spearman_abs"].clip(0, 0.3) / 0.3)
_final["s_mi"]     = _final["MI_norm"].fillna(0)
_final["s_nan"]    = 1 - _final["nan_rate"].clip(0, 1)        # NaN 少ない = 高スコア
_final["s_leak"]   = _final["leak_risk"].map({"HIGH":0,"MEDIUM":0.5,"LOW":1.0}).fillna(1.0)
_final["s_vif"]    = (1 / (_final["VIF"].clip(1, 50) / 50 + 1e-6)
                      ).pipe(_s01).fillna(0.5)               # VIF 小さい = 高スコア

# 総合スコア: 重み付け平均
_final["total"] = (
    0.40 * _final["s_model"] +
    0.20 * _final["s_univ"]  +
    0.10 * _final["s_mi"]    +
    0.15 * _final["s_nan"]   +
    0.10 * _final["s_leak"]  +
    0.05 * _final["s_vif"]
).clip(0, 1)

# ── S/A/B/C ランク ────────────────────────────────────────────────────────
def _rank(s):
    if s >= 0.65: return "S"
    if s >= 0.45: return "A"
    if s >= 0.25: return "B"
    return "C"

_final["rank"] = _final["total"].apply(_rank)
_final = _final.sort_values("total", ascending=False).reset_index(drop=True)
_final["#"] = _final.index + 1

# ── 運用分類 ─────────────────────────────────────────────────────────────
def _op_class(row):
    if row["leak_risk"] == "HIGH":                 return "削除候補(リーク)"
    if row["断捨離アクション"].startswith("削除推奨"): return "削除候補(高NaN)"
    if row["nan_rate"] > 0.5 or row["rank"] == "C": return "削除候補"
    if row["rank"] == "S" and row["nan_rate"] < 0.05: return "必須"
    if row["rank"] in ("S","A"):                    return "推奨"
    return "保持検討"

_final["運用分類"] = _final.apply(_op_class, axis=1)

# ── ランク集計 ────────────────────────────────────────────────────────────
print(f"{'='*65}")
print(f"【⑦ 最終評価サマリー】  total_score = モデル重要度×0.4 + 単変量×0.2 + MI×0.1 + データ品質×0.3")
print(f"{'='*65}")
_rc = {r: (_final["rank"]==r).sum() for r in ["S","A","B","C"]}
print(f"  S ランク (≥0.65): {_rc['S']:3d} 列  ← 最重要: 削除するとモデル精度が大きく低下")
print(f"  A ランク (≥0.45): {_rc['A']:3d} 列  ← 重要: 保持推奨")
print(f"  B ランク (≥0.25): {_rc['B']:3d} 列  ← 中程度: 状況次第で削除検討")
print(f"  C ランク (< 0.25): {_rc['C']:3d} 列  ← 低寄与: 削除候補")
print()
_oc = _final["運用分類"].value_counts()
for _op in ["必須","推奨","保持検討","削除候補","削除候補(高NaN)","削除候補(リーク)"]:
    if _op in _oc: print(f"  {_op:16s}: {_oc[_op]:3d} 列")

# ── 可視化 ────────────────────────────────────────────────────────────────
fig_l6, axes_l6 = plt.subplots(1, 3, figsize=(19, 10))
_rcols = {"S":"#1f77b4","A":"#2ca02c","B":"#ff7f0e","C":"#d62728"}

# 左: TOP 45 横棒グラフ
_top45 = _final.head(45)
_cb45  = [_rcols[r] for r in _top45["rank"]]
axes_l6[0].barh(_top45["feature"][::-1], _top45["total"][::-1], color=_cb45[::-1])
_legend_h = [Patch(color=c, label=r) for r, c in _rcols.items()]
axes_l6[0].legend(handles=_legend_h, loc="lower right", fontsize=8)
axes_l6[0].set_xlabel("総合スコア (0〜1)")
axes_l6[0].set_title(f"総合スコア TOP 45\n(S={_rc['S']} / A={_rc['A']} / B={_rc['B']} / C={_rc['C']})")
axes_l6[0].axvline(0.65, color="steelblue", ls="--", alpha=0.4)
axes_l6[0].axvline(0.45, color="green",     ls="--", alpha=0.4)

# 中: 運用分類別パイチャート
_op_order_l6 = ["必須","推奨","保持検討","削除候補","削除候補(高NaN)","削除候補(リーク)"]
_op_colors   = ["#1f77b4","#2ca02c","#ff7f0e","#d62728","#9467bd","#8c564b"]
_op_vals  = [_oc.get(o, 0) for o in _op_order_l6]
_op_nonz  = [(o, v, c) for o, v, c in zip(_op_order_l6, _op_vals, _op_colors) if v > 0]
axes_l6[1].pie([v for _,v,_ in _op_nonz],
               labels=[f"{o}\n({v})" for o,v,_ in _op_nonz],
               colors=[c for _,_,c in _op_nonz],
               autopct="%1.0f%%", startangle=140, textprops={"fontsize": 8})
axes_l6[1].set_title("運用分類内訳")

# 右: ランク × NaN率 散布図
_c_sc = [_rcols[r] for r in _final["rank"]]
axes_l6[2].scatter(_final["nan_rate"]*100, _final["total"], c=_c_sc, alpha=0.7, s=40)
axes_l6[2].axhline(0.65, color="steelblue", ls="--", alpha=0.4, label="S閾値 0.65")
axes_l6[2].axhline(0.45, color="green",     ls="--", alpha=0.4, label="A閾値 0.45")
axes_l6[2].axvline(50,   color="red",       ls="--", alpha=0.3, label="NaN>50%")
axes_l6[2].set_xlabel("NaN 率 (%)")
axes_l6[2].set_ylabel("総合スコア")
axes_l6[2].set_title("スコア vs NaN率  (色=ランク)")
axes_l6[2].legend(fontsize=8)
_legend_h2 = [Patch(color=c, label=r) for r, c in _rcols.items()]
axes_l6[2].legend(handles=_legend_h2, fontsize=8)

plt.suptitle("⑦ 最終評価 — S/A/B/C ランク + 必須 / 推奨 / 削除候補", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# ── 詳細テーブル ─────────────────────────────────────────────────────────
_cols_show = ["#","feature","rank","total","nan_rate","spearman","perm_imp","運用分類"]

print("\n" + "="*70)
print("【必須特徴量 (S ランク / NaN < 5%)】")
_ess = _final[_final["運用分類"]=="必須"]
if len(_ess):
    display(_ess[_cols_show].to_string(index=False))
else:
    print("  (なし — Sランクでも NaN ≥ 5% の列があります)")

print("\n【推奨特徴量 (A/S ランク)】")
display(_final[_final["運用分類"]=="推奨"][_cols_show].to_string(index=False))

print("\n【削除候補特徴量】")
_drop = _final[_final["運用分類"].str.startswith("削除")]
_cols_drop = ["#","feature","rank","total","nan_rate","leak_risk","運用分類","断捨離アクション"]
display(_drop[_cols_drop].to_string(index=False))

# ── 保存 ─────────────────────────────────────────────────────────────────
final_rank_df = _final.copy()
_out_path = ROOT / "docs" / "reports" / "feature_rank_report.csv"
_out_path.parent.mkdir(parents=True, exist_ok=True)
final_rank_df.to_csv(_out_path, index=False, encoding="utf-8-sig")
print(f"\n✓ final_rank_df 保存: {_out_path.name}  ({len(final_rank_df)} 列)")

---
## LLM 特徴量考察レポート生成

**Python 数値計算 → JSON → OSS LLM → 解釈・改善案** のパイプライン実装。

```
Cell M1  JSON 構築・保存    feature_analysis.json を生成
Cell M2  LLM 設定           モデル・エンドポイント・プロンプトを設定
Cell M3  API 呼び出し       Ollama / OpenAI 互換 API で考察レポートを生成
```

**対応 LLM バックエンド**

| バックエンド | 設定 | 推奨モデル |
|---|---|---|
| Ollama (ローカル) | `LLM_BACKEND = "ollama"` | `qwen3:14b` / `qwen3:32b` / `llama3.3` |
| OpenAI 互換 (クラウド) | `LLM_BACKEND = "openai"` | Groq / Together.ai / OpenRouter |
| プロンプト出力のみ | `LLM_BACKEND = "print"` | — (コピー & ペーストで手動入力) |

In [ ]:
## ── Cell P_ROI: ROI・的中率分析 ─────────────────────────────────────────
# テストセット上で Top1 / Top3 ヒット率・回収率 (ROI) を計算する

# ── テストセット = _df_for_split の test 行 ──────────────────────────────
# implied_prob は UNNECESSARY_COLUMNS に含まれるため X_test_nb には無い
# → _df_for_split (add_derived_features 後・prepare 前) から取得する
_te_df = _df_for_split[_te_mask_nb].reset_index(drop=True)
_tm    = _te_df[["race_id", "finish"]].copy()
_tm["pred"] = model_nb.predict(X_test_nb)

# implied_prob / odds から1着理論配当を算出
if "implied_prob" in _te_df.columns:
    _tm["implied_prob"] = _te_df["implied_prob"].values
elif "odds" in _te_df.columns:
    _tm["implied_prob"] = (1.0 / _te_df["odds"].clip(lower=1.0)).values
else:
    _tm["implied_prob"] = None   # オッズ情報なし

_has_finish = _tm["finish"].notna().any()

if not _has_finish:
    print("⚠ finish 列が取得できません → roi_stats をスキップ")
    roi_stats = {"note": "finish column not available"}
else:
    # ── レースごとに予測スコアで順位付け ────────────────────────────────
    _tm["pred_rank"] = (
        _tm.groupby("race_id")["pred"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    _n_races  = int(_tm["race_id"].nunique())
    _top1_df  = _tm[_tm["pred_rank"] == 1].copy()

    # 的中カウント
    _top1_win  = int((_top1_df["finish"] == 1).sum())
    _top1_show = int((_top1_df["finish"] <= 3).sum())
    _top3_contains_winner = int(
        _tm[_tm["pred_rank"] <= 3]
        .groupby("race_id")["finish"].min()
        .pipe(lambda s: (s == 1).sum())
    )

    # ── ROI 計算 (implied_prob が取得できた場合のみ) ─────────────────────
    _roi_pct = None
    _avg_ip  = None
    if _tm["implied_prob"].notna().any():
        _top1_df["_payout"] = np.where(
            _top1_df["finish"] == 1,
            100 / _top1_df["implied_prob"].fillna(0.1).clip(lower=0.02),
            0.0
        )
        _roi_pct = float(_top1_df["_payout"].sum() / (_n_races * 100) - 1) * 100
        _avg_ip  = round(float(_top1_df["implied_prob"].mean()), 4)

    roi_stats = {
        "n_races":          _n_races,
        "top1_win_rate":    round(_top1_win  / _n_races, 4),
        "top1_show_rate":   round(_top1_show / _n_races, 4),
        "top3_hit_rate":    round(_top3_contains_winner / _n_races, 4),
        "roi_top1_pct":     round(_roi_pct, 2) if _roi_pct is not None else None,
        "avg_implied_prob": _avg_ip,
    }

    # ── 可視化 ────────────────────────────────────────────────────────────
    _ncols = 2 if _roi_pct is not None else 1
    fig_roi, _axes_roi = plt.subplots(1, _ncols, figsize=(7 * _ncols, 4))
    if _ncols == 1: _axes_roi = [_axes_roi]

    _avg_field = 1 / _tm.groupby("race_id").size().mean()
    _hit_vals  = {
        f"単勝的中\n(Top1→1着)\nランダム {_avg_field*100:.1f}%":
            roi_stats["top1_win_rate"],
        "複勝的中\n(Top1→3着内)":
            roi_stats["top1_show_rate"],
        "Top3包含\n(3頭に1着)":
            roi_stats["top3_hit_rate"],
    }
    _axes_roi[0].bar(list(_hit_vals.keys()), list(_hit_vals.values()),
                     color=["#4C78A8", "#54A24B", "#F58518"])
    _axes_roi[0].axhline(_avg_field, color="red", ls="--", alpha=0.5)
    for _ii, _vv in enumerate(_hit_vals.values()):
        _axes_roi[0].text(_ii, _vv + 0.003, f"{_vv*100:.1f}%", ha="center", fontsize=10, fontweight="bold")
    _axes_roi[0].set_ylabel("的中率"); _axes_roi[0].set_title(f"的中率  ({_n_races:,} レース)")

    if _roi_pct is not None:
        _c_roi = "#54A24B" if _roi_pct > -25 else "#E45756"
        _axes_roi[1].bar(["ROI (理論単勝)"], [_roi_pct], color=_c_roi, width=0.4)
        _axes_roi[1].axhline(0,   color="black",  ls="-",  lw=1)
        _axes_roi[1].axhline(-25, color="orange", ls="--", alpha=0.6, label="JRA 控除 -25%")
        _axes_roi[1].text(0, _roi_pct + 1.0, f"{_roi_pct:+.1f}%", ha="center", fontsize=13, fontweight="bold")
        _axes_roi[1].set_ylabel("ROI (%)"); _axes_roi[1].set_title("回収率 ROI")
        _axes_roi[1].legend(fontsize=8)

    plt.suptitle("ROI・的中率分析  (テストセット)", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

    print(f"\n{'='*55}")
    print(f"【ROI・的中率サマリー ({_n_races:,} レース)】")
    print(f"  単勝的中率  (Top1→1着):  {roi_stats['top1_win_rate']*100:.1f}%  (ランダム {_avg_field*100:.1f}%)")
    print(f"  複勝的中率  (Top1→3着内): {roi_stats['top1_show_rate']*100:.1f}%")
    print(f"  Top3 包含率:              {roi_stats['top3_hit_rate']*100:.1f}%")
    if _roi_pct is not None:
        print(f"  ROI (理論単勝):           {_roi_pct:+.1f}%")
    else:
        print(f"  ROI: オッズデータなしのため計算不可")
    print(f"\n✓ roi_stats 構築完了")

In [ ]:
## ── Cell M1: 分析結果 JSON 構築 + 保存 ─────────────────────────────────
import json, math

def _to_py(v):
    if isinstance(v, float) and (math.isnan(v) or math.isinf(v)): return None
    if hasattr(v, "item"): return v.item()
    return v

# ── グループマッピング ────────────────────────────────────────────────────
_feat_to_grp = {_c: _gn for _gn, _gcs in _GMAP.items() for _c in _gcs}

# ── 特徴量ごとの統合データ ────────────────────────────────────────────────
_fa_features = []
for _, row in final_rank_df.iterrows():
    c = row["feature"]
    _fa_features.append({
        "rank":        _to_py(row["#"]),
        "feature":     c,
        "group":       _feat_to_grp.get(c, "⑪その他"),
        "tier":        row["rank"],
        "op_class":    row["運用分類"],
        "total_score": round(_to_py(row["total"]) or 0, 4),
        "spearman":    round(_to_py(row["spearman"]) or 0, 4),
        "MI_norm":     round(_to_py(row.get("MI_norm", 0)) or 0, 3),
        "gain_norm":   round(_to_py(row.get("gain_norm", 0)) or 0, 4),
        "perm_imp":    round(_to_py(row["perm_imp"]) or 0, 4),
        "shap":        round(_to_py(row.get("shap", 0)) or 0, 4) if _shap_ok else None,
        "nan_rate":    round(_to_py(row["nan_rate"]) or 0, 3),
        "leak_risk":   row["leak_risk"],
        "vif":         round(_to_py(row.get("VIF", 1)) or 1, 1),
    })

# ── 高相関ペア (TOP 30) ──────────────────────────────────────────────────
_hc_list = [
    {"A": r["A"], "B": r["B"], "r": round(_to_py(r["r"]), 4)}
    for _, r in high_corr_df.head(30).iterrows()
]

# ── グループサマリー ─────────────────────────────────────────────────────
_gain_total = group_df["Gain合計"].sum() or 1
_grp_list = []
for _, r in group_df.iterrows():
    _grp_list.append({
        "group":             r["グループ"],
        "n_features":        int(r["列数"]),
        "mean_nan_rate":     round(float(r["平均NaN率"]), 3),
        "mean_spearman_abs": round(float(r["平均|Spearman|"]), 4),
        "gain_share_pct":    round(float(r["Gain合計"]) / _gain_total * 100, 1),
        "ablation_delta_sp": round(float(r["Perm合計ΔSp"]), 4),
    })

# ── サマリー ─────────────────────────────────────────────────────────────
def _feat_list(op): return final_rank_df[final_rank_df["運用分類"] == op]["feature"].tolist()

_summary = {
    "S_rank":                final_rank_df[final_rank_df["rank"] == "S"]["feature"].tolist(),
    "A_rank":                final_rank_df[final_rank_df["rank"] == "A"]["feature"].tolist(),
    "B_rank_count":          int((final_rank_df["rank"] == "B").sum()),
    "C_rank_count":          int((final_rank_df["rank"] == "C").sum()),
    "essential":             _feat_list("必須"),
    "recommended":           _feat_list("推奨"),
    "delete_high_nan":       _feat_list("削除候補(高NaN)"),
    "delete_general":        _feat_list("削除候補"),
    "delete_leak":           _feat_list("削除候補(リーク)"),
    "near_zero_corr":        univariate_df[univariate_df["spearman_abs"] < 0.01]["feature"].tolist(),
    "high_corr_pairs_count": len(high_corr_df),
}

# ── LightGBM メタ ─────────────────────────────────────────────────────────
_meta = {
    "target":            TRAIN_TARGET,
    "date_range":        f"{DATE_START} 〜 {DATE_END}",
    "n_features":        len(_feature_cols_nb),
    "n_train":           int(len(X_train_nb)),
    "n_test":            int(len(X_test_nb)),
    "baseline_spearman": round(float(_metrics_nb.get("spearman", 0)), 4),
    "baseline_rmse":     round(float(_metrics_nb.get("rmse", 0)), 4),
    "optuna_trials":     OPTUNA_TRIALS if USE_OPTUNA else 0,
    "shap_computed":     _shap_ok,
    "model_type":        "LightGBM regression (speed_deviation)",
}

# ── JSON 組み立て (ROI 含む) ──────────────────────────────────────────────
feature_analysis_json = {
    "meta":            _meta,
    "roi_stats":       roi_stats if "roi_stats" in dir() else {},  # Cell P_ROI の結果
    "features":        _fa_features,
    "high_corr_pairs": _hc_list,
    "groups":          _grp_list,
    "summary":         _summary,
}

# ── 保存 ─────────────────────────────────────────────────────────────────
_json_path = ROOT / "docs" / "reports" / "feature_analysis.json"
_json_path.parent.mkdir(parents=True, exist_ok=True)
with open(_json_path, "w", encoding="utf-8") as _f:
    json.dump(feature_analysis_json, _f, ensure_ascii=False, indent=2)

_js = _json_path.stat().st_size
print(f"✓ feature_analysis.json 保存完了  ({_js/1024:.1f} KB)")
print(f"  特徴量: {len(_fa_features)} 列  |  高相関ペア: {len(_hc_list)}  |  ROI: {feature_analysis_json['roi_stats']}")

In [ ]:
## ── Cell P_Ollama: Ollama セットアップ確認 ──────────────────────────────
import requests as _req, subprocess, os, shutil

_OLLAMA_BASE   = "http://localhost:11434"
_TARGET_MODEL  = "qwen3:14b"

# Ollama CLI パスを探す (Windows では PATH 外にインストールされることがある)
_default_exe = os.path.join(os.environ.get("LOCALAPPDATA",""), "Programs", "Ollama", "ollama.exe")
_OLLAMA_EXE  = shutil.which("ollama") or (_default_exe if os.path.isfile(_default_exe) else None)

print("【Ollama セットアップ確認】\n")

# ── サーバー疎通 ────────────────────────────────────────────────────────
try:
    _resp = _req.get(f"{_OLLAMA_BASE}/api/tags", timeout=3)
    _resp.raise_for_status()
    _avail = [m["name"] for m in _resp.json().get("models", [])]
    print(f"✓ Ollama サーバー起動中  ({_OLLAMA_BASE})")

    if _avail:
        print("  インストール済みモデル:")
        for _m in _avail:
            _mark = "  ← ✓ 使用予定" if _TARGET_MODEL in _m else ""
            print(f"    {_m}{_mark}")
    else:
        print("  (モデル未インストール)")

    _model_ready = any(_TARGET_MODEL in _m for _m in _avail)

    if _model_ready:
        print(f"\n✅ {_TARGET_MODEL} 利用可能 → Cell M3 をそのまま実行できます")
    else:
        # ダウンロード中かどうかを ps コマンドで確認
        _downloading = False
        if _OLLAMA_EXE:
            try:
                _ps = subprocess.run(
                    [_OLLAMA_EXE, "ps"], capture_output=True, text=True, timeout=5
                )
                if _TARGET_MODEL in _ps.stdout:
                    _downloading = True
                    print(f"\n⏳ {_TARGET_MODEL} ダウンロード中です…")
                    print(_ps.stdout.strip())
            except Exception:
                pass

        if not _downloading:
            print(f"\n⚠ {_TARGET_MODEL} が未ダウンロードです")
            _pull_cmd = f'"{_OLLAMA_EXE}"' if _OLLAMA_EXE else "ollama"
            print(f"   PowerShell で実行してください:")
            print(f"   > {_pull_cmd} pull {_TARGET_MODEL}  (約 9GB)")

except _req.exceptions.ConnectionError:
    print("❌ Ollama サーバーが起動していません\n")
    print("━" * 55)
    print("【インストール・起動手順】\n")
    print("STEP 1  Ollama をインストール")
    print("        https://ollama.com/download  (Windows 版)")
    print()
    print("STEP 2  タスクトレイの Ollama アイコンを確認")
    print("        (インストール後は自動起動します)")
    print()
    _pull_exe = f'"{_OLLAMA_EXE}"' if _OLLAMA_EXE else "ollama"
    print("STEP 3  別の PowerShell タブでモデルをダウンロード")
    print(f"        > {_pull_exe} pull {_TARGET_MODEL}   (約 9GB ・推奨)")
    print(f"        > {_pull_exe} pull qwen3:32b          (約 20GB・高精度)")
    print()
    print("STEP 4  このセルを再実行して確認 → Cell M3 を実行")
    print("━" * 55)
    print()
    print("💡 Groq 代替手段 (無料・即座):")
    print("   https://console.groq.com でキー取得後、Cell M2 を変更:")
    print("     LLM_BACKEND  = 'openai'")
    print("     LLM_MODEL    = 'llama-3.3-70b-versatile'")
    print("     LLM_API_BASE = 'https://api.groq.com/openai/v1'")
    print("     LLM_API_KEY  = 'gsk_xxxx...'  # 取得したキー")

In [ ]:
## ── Cell M2: LLM 設定 + プロンプト生成 ─────────────────────────────────
import requests as _req2

# ── ① バックエンド選択 ───────────────────────────────────────────────────
LLM_BACKEND = "ollama"     # "ollama" | "openai" | "print"

# ── ② モデル優先順位 (上から順に利用可能なものを自動選択) ─────────────────
_MODEL_PRIORITY = [
    "qwen3:14b",   # 高品質 (9GB)
    "qwen3:4b",    # 軽量・高速 (2.5GB)
    "qwen3:1.7b",  # 超軽量 (1GB)
]
LLM_MODEL = _MODEL_PRIORITY[0]

if LLM_BACKEND == "ollama":
    try:
        _tags = _req2.get("http://localhost:11434/api/tags", timeout=2).json()
        _installed = [m["name"] for m in _tags.get("models", [])]
        for _cand in _MODEL_PRIORITY:
            if any(_cand in _m for _m in _installed):
                LLM_MODEL = _cand
                break
        if _installed:
            print(f"  インストール済みモデル: {_installed}")
    except Exception:
        pass

# ── ③ エンドポイント ──────────────────────────────────────────────────────
LLM_API_BASE = "http://localhost:11434/v1"
LLM_API_KEY  = "ollama"

# ── ④ 分析内容 ───────────────────────────────────────────────────────────
LLM_ANALYSIS_ITEMS = {
    "feature_ranking":    True,
    "delete_candidates":  True,
    "leakage_check":      True,
    "multicollinearity":  True,
    "class_features":     True,
    "popularity_bias":    True,
    "roi_features":       True,
    "feature_suggestion": True,
}

# ── プロンプト組み立て ────────────────────────────────────────────────────
def _build_compact_json(fa: dict, max_features: int = 50) -> str:
    _top = fa["features"][:max_features]
    _fc  = [{
        "no": f["rank"], "feature": f["feature"], "group": f["group"],
        "tier": f["tier"], "op": f["op_class"],
        "sp": f["spearman"], "MI": f["MI_norm"],
        "gain": f["gain_norm"], "perm": f["perm_imp"],
        "shap": f["shap"], "nan": f["nan_rate"], "leak": f["leak_risk"],
    } for f in _top]
    return json.dumps({
        "meta":            fa["meta"],
        "roi_stats":       fa.get("roi_stats", {}),
        "features":        _fc,
        "high_corr_top20": fa["high_corr_pairs"][:20],
        "groups":          fa["groups"],
        "summary":         fa["summary"],
    }, ensure_ascii=False, separators=(",", ":"))

_json_for_llm = _build_compact_json(feature_analysis_json)

_item_labels = {
    "feature_ranking":    "## 1. 特徴量重要度ランキング\nSpearman・SHAP・Gain を統合した TOP 20 を表形式で示し、各特徴量の役割を説明してください。",
    "delete_candidates":  "## 2. 削除候補特徴量\nNaN率・重要度・多重共線性を踏まえ、削除すべき特徴量を表形式でリストし、削除理由と代替案を記載してください。",
    "leakage_check":      "## 3. データリーク疑惑特徴量\n`leak_risk=MEDIUM` 以上の列の運用リスクを表形式で説明してください。",
    "multicollinearity":  "## 4. 多重共線性疑惑特徴量\n高相関ペア(|r|≥0.85)の冗長列を特定し、どちらを残すか根拠とともに推奨してください。",
    "class_features":     "## 5. クラス情報特徴量の評価\n`race_class_num`・`class_change` 等の有効性と改善案を示してください。",
    "popularity_bias":    "## 6. 人気依存の評価\nオッズ・人気系特徴量がモデルに与える影響と過学習リスクを評価してください。",
    "roi_features":       "## 7. ROI・回収率向上に有効な特徴量\nroi_stats の数値を踏まえ、高オッズ馬の妙味検出に寄与しやすい特徴量を推定してください。",
    "feature_suggestion": "## 8. 追加すべき特徴量の提案\n現状の弱点を補う特徴量を 5〜8 件、追加方法とともに提案してください。",
}
_analysis_request = "\n\n".join(
    v for k, v in _item_labels.items() if LLM_ANALYSIS_ITEMS.get(k)
)

# ── ROI サマリー ──────────────────────────────────────────────────────────
_roi_context = ""
_rs = feature_analysis_json.get("roi_stats", {})
if _rs and "top1_win_rate" in _rs:
    _roi_context = f"""
## ROI 実績（テストセット {_rs.get('n_races', '?')} レース）
| 指標 | 値 |
|---|---|
| 単勝的中率 (Top1→1着) | {_rs['top1_win_rate']*100:.1f}% |
| 複勝的中率 (Top1→3着内) | {_rs['top1_show_rate']*100:.1f}% |
| Top3 包含率 | {_rs['top3_hit_rate']*100:.1f}% |
| ROI（理論単勝） | {_rs['roi_top1_pct']:+.1f}%（JRA 控除 -25% が実運用基準） |
"""

# ── システムプロンプト（日本語強制） ─────────────────────────────────────
SYSTEM_PROMPT = """/no_think
あなたは日本競馬の機械学習システムにおける特徴量分析の専門家です。

【厳守事項】
- 必ず日本語のみで回答すること。英語は一切使用禁止。
- 「Okay」「Let me」「First」「I need to」などの英語前置き・思考過程を出力しないこと。
- 「## 1.」から直接回答を開始すること。前置き・まとめ不要。
- 各セクションは必ずMarkdown見出し（##）で始め、表形式で整理すること。

【背景】
- データ期間: 2013〜2018年
- 目的変数 `speed_deviation` = 各馬の速度と同レース平均の差（標準化）
- Spearman ≈ 0.28（競馬予測としては妥当な水準）
- 日本競馬では人気・オッズの相関が高く、人気依存に注意が必要"""

USER_PROMPT = f"""/no_think

以下のデータを分析し、各セクション（## 1. 〜 ## 8.）を日本語の表形式で回答してください。
英語は使用しないこと。「## 1.」から直接回答を開始してください。
{_roi_context}
```json
{_json_for_llm}
```

{_analysis_request}
"""

print(f"プロンプト構築完了")
print(f"  System : {len(SYSTEM_PROMPT):,} 文字 | User : {len(USER_PROMPT):,} 文字 (JSON {len(_json_for_llm)/1024:.1f} KB)")
print(f"  Backend: {LLM_BACKEND} / Model: {LLM_MODEL}")

In [ ]:
## ── Cell M3: レポート生成（gen_feature_report.py 実行 + LLM 総括コメント）──
# docs/reports/gen_feature_report.py でテーブルを Python 生成し、
# LLM は短い総括コメントのみ担当する。
#
# 特徴量の説明は docs/reports/feature_descriptions.yaml で管理。
# 新しい特徴量を追加したら YAML に一行追記するだけでレポートに反映されます。

import subprocess, sys, time, re as _re, requests
from IPython.display import Markdown, display as _disp

_t_start = time.time()

# ── 1. Python スクリプトでレポート生成 ──────────────────────────────────
_script = ROOT / "docs" / "reports" / "gen_feature_report.py"
_out_path_md = ROOT / "docs" / "reports" / "feature_llm_report.md"
if _script.exists():
    _result = subprocess.run([sys.executable, str(_script)],
                             capture_output=True, text=True, encoding="utf-8")
    print(_result.stdout.strip())
    if _result.returncode != 0:
        print("ERROR: gen_feature_report.py 失敗")
        print(_result.stderr)
        raise RuntimeError("gen_feature_report.py failed")
    _report_text = _out_path_md.read_text(encoding="utf-8")
else:
    print(f"WARNING: {_script} not found; skipping report generation")
    if _out_path_md.exists():
        _report_text = _out_path_md.read_text(encoding="utf-8")
    else:
        _report_text = "# 特徴量分析レポート\n\n`gen_feature_report.py` が見つからないため、生成処理をスキップしました。"
        _out_path_md.write_text(_report_text, encoding="utf-8")
print(f"  読み込み: {len(_report_text):,} 文字")
# ── 3. LLM 総括コメント（オプション） ────────────────────────────────────
def _is_mostly_english(text, threshold=0.55):
    ascii_cnt = sum(1 for c in text if ord(c) < 128 and c.isprintable())
    return ascii_cnt / max(len(text), 1) > threshold

def _first_japanese_para(text):
    for i, line in enumerate(text.split("\n")):
        s = line.strip()
        if s.startswith("#") or _re.search(r"[\u3040-\u9FFF]", s) or \
                (s.startswith("-") and _re.search(r"[\u3040-\u9FFF]", s)):
            return "\n".join(text.split("\n")[i:]).strip()
    return text.strip()

if LLM_BACKEND != "print":
    print(f"LLM ({LLM_MODEL}) で総括コメント生成中…")
    try:
        _fa   = feature_analysis_json
        _rs   = _fa.get("roi_stats", {})
        _meta = _fa.get("meta", {})
        _feats = _fa.get("features", [])
        _corrs = _fa.get("high_corr_pairs", [])
        _leak_feats = [f for f in _feats if f.get("leak_risk","") in ("HIGH","MEDIUM")]
        _del_feats  = [f for f in _feats if f.get("tier") in ("C","D")]

        _sys_cmt = "/no_think\n競馬AIの特徴量改善コンサルタント。日本語で短く技術的なアドバイスをする。"
        _usr_cmt = (
            f"/no_think\n\n以下のモデル指標を見て改善ポイントを箇条書き3〜5点で述べてください（日本語のみ、表不要）。\n\n"
            f"- Spearman相関: {_meta.get('spearman','?')}\n"
            f"- 特徴量数: {_meta.get('n_features','?')}\n"
            f"- ROI(理論単勝): {_rs.get('roi_top1_pct','?')}%\n"
            f"- 削除候補特徴量: {len(_del_feats)}件\n"
            f"- 高相関ペア: {len(_corrs)}件\n"
            f"- リーク疑惑: {len(_leak_feats)}件\n"
        )
        if LLM_BACKEND == "ollama":
            _cmt_raw = requests.post(
                "http://localhost:11434/api/chat",
                json={"model": LLM_MODEL, "stream": False, "think": False,
                      "options": {"temperature": 0.1, "num_predict": 1024},
                      "messages": [{"role":"system","content":_sys_cmt},
                                   {"role":"user","content":_usr_cmt}]},
                timeout=120,
            ).json()["message"].get("content","").strip()
        else:
            _cmt_raw = requests.post(
                f"{LLM_API_BASE.rstrip('/')}/chat/completions",
                headers={"Authorization": f"Bearer {LLM_API_KEY}","Content-Type":"application/json"},
                json={"model":LLM_MODEL,"temperature":0.1,"max_tokens":1024,
                      "messages":[{"role":"system","content":_sys_cmt},{"role":"user","content":_usr_cmt}]},
                timeout=120,
            ).json()["choices"][0]["message"]["content"]

        _cmt = _first_japanese_para(_cmt_raw)
        if _cmt and not _is_mostly_english(_cmt):
            _report_text += f"\n\n---\n\n## 🔍 総括コメント（{LLM_MODEL}）\n\n{_cmt}"
            _out_path_md.write_text(_report_text, encoding="utf-8")
            print(f"  ✓ コメント追加・上書き保存: {len(_cmt)} 文字")
        else:
            print(f"  ⚠ コメントが英語主体のためスキップ")
    except Exception as _ce:
        print(f"  ⚠ LLM コメント取得失敗 (スキップ): {_ce}")

# ── 4. 表示 ──────────────────────────────────────────────────────────────
_elapsed = time.time() - _t_start
print(f"\n✓ 完了 ({_elapsed:.1f}s) | {len(_report_text):,} 文字 | {_out_path_md.name}")
_disp(Markdown(f"---\n## 📊 特徴量分析レポート\n\n{_report_text}"))

In [ ]:
## ── Cell P_Notion: Notion エクスポート ──────────────────────────────────
# LLM が生成したレポートを Notion ページとして保存する

# ── 設定 ─────────────────────────────────────────────────────────────────
import os
NOTION_TOKEN   = os.getenv("NOTION_TOKEN", "")
NOTION_PAGE_ID = "https://www.notion.so/AI-37e2e86b510e804cbad3df2c0f322f68"
# ↑ URL でも UUID でもそのまま貼ってOK

# ── URL → UUID 変換 ───────────────────────────────────────────────────────
import re as _re_n
def _extract_notion_uuid(raw: str) -> str:
    _s = raw.strip().split("?")[0].split("#")[0]
    if _re_n.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", _s, _re_n.I):
        return _s.lower()
    _m = _re_n.search(r"([0-9a-f]{32})$", _s, _re_n.I)
    if _m:
        _h = _m.group(1).lower()
        return f"{_h[:8]}-{_h[8:12]}-{_h[12:16]}-{_h[16:20]}-{_h[20:]}"
    return _s

_page_id = _extract_notion_uuid(NOTION_PAGE_ID)
print(f"Page ID (変換後): {_page_id}")

# ── notion-client インストール確認 ────────────────────────────────────────
import sys
try:
    from notion_client import Client as _NC
except ImportError:
    import subprocess
    print("notion-client をインストール中...")
    subprocess.run([sys.executable, "-m", "pip", "install", "notion-client", "-q"], check=True)
    from notion_client import Client as _NC

# ── rich_text 生成 (太字・コード・インラインコード対応) ────────────────────
def _rt(text: str) -> list:
    """Markdown の **太字** と `コード` をパースして rich_text リストを返す"""
    result = []
    # **bold** と `code` を交互にパース
    pattern = _re_n.compile(r"(\*\*(.+?)\*\*|`(.+?)`)")
    pos = 0
    for m in pattern.finditer(text):
        # 前のプレーンテキスト
        if m.start() > pos:
            plain = text[pos:m.start()][:2000]
            if plain:
                result.append({"type": "text", "text": {"content": plain}})
        if m.group(0).startswith("**"):
            result.append({"type": "text", "text": {"content": m.group(2)[:200]},
                           "annotations": {"bold": True}})
        else:  # `code`
            result.append({"type": "text", "text": {"content": m.group(3)[:200]},
                           "annotations": {"code": True}})
        pos = m.end()
    # 残りのテキスト
    tail = text[pos:][:2000]
    if tail:
        result.append({"type": "text", "text": {"content": tail}})
    return result or [{"type": "text", "text": {"content": text[:2000]}}]

# ── セクションアイコンマップ ──────────────────────────────────────────────
_SECTION_ICONS = {
    "1": "🏆", "2": "🗑️", "3": "⚠️", "4": "🔗",
    "5": "🏇", "6": "📈", "7": "💰", "8": "💡",
}
_SECTION_COLORS = {
    "1": "blue", "2": "red", "3": "orange", "4": "purple",
    "5": "green", "6": "yellow", "7": "pink", "8": "gray",
}

def _section_num(heading: str) -> str:
    """見出しテキストから番号を抽出 ('1. ...' → '1')"""
    m = _re_n.match(r"\s*(\d+)", heading)
    return m.group(1) if m else ""

# ── Markdown → Notion ブロック変換 ────────────────────────────────────────
def _md_to_blocks(md: str) -> list:
    blocks     = []
    tbl_rows   = []
    in_table   = False
    in_code    = False
    code_lines = []
    prev_was_h2 = False

    def _flush_table():
        if not tbl_rows: return
        w = max(len(r) for r in tbl_rows)
        children = []
        for row in tbl_rows:
            cells = [[{"type": "text", "text": {"content": c.strip()[:200]}}]
                     for c in (row + [""] * w)[:w]]
            children.append({
                "object": "block", "type": "table_row",
                "table_row": {"cells": cells},
            })
        blocks.append({
            "object": "block", "type": "table",
            "table": {"table_width": w, "has_column_header": True,
                      "has_row_header": False, "children": children},
        })
        tbl_rows.clear()

    def _flush_code(lang=""):
        if not code_lines: return
        blocks.append({
            "object": "block", "type": "code",
            "code": {
                "rich_text": [{"type": "text", "text": {"content": "\n".join(code_lines)[:2000]}}],
                "language": lang or "plain text",
            },
        })
        code_lines.clear()

    for line in md.split("\n"):
        # コードフェンス処理
        if line.strip().startswith("```"):
            if in_code:
                _flush_code()
                in_code = False
            else:
                if in_table: _flush_table(); in_table = False
                lang = line.strip()[3:].strip()
                in_code = True
            continue
        if in_code:
            code_lines.append(line)
            continue

        # テーブル行
        if line.strip().startswith("|") and "|" in line[1:]:
            cols = [c.strip() for c in line.strip().strip("|").split("|")]
            # セパレータ行 (|---|) を除外
            if not all(_re_n.fullmatch(r"[:\-\s]+", c) for c in cols):
                tbl_rows.append(cols)
            in_table = True
            continue
        if in_table:
            _flush_table(); in_table = False

        s = line.strip()

        # 空行
        if not s:
            continue

        # H1
        if line.startswith("# ") and not line.startswith("## "):
            heading_text = line[2:].strip()
            blocks.append({"object":"block","type":"heading_1",
                            "heading_1":{"rich_text":_rt(heading_text)}})
            prev_was_h2 = False

        # H2 → divider + callout で見やすくする
        elif line.startswith("## "):
            heading_text = line[3:].strip()
            num = _section_num(heading_text)
            icon = _SECTION_ICONS.get(num, "📌")
            color = _SECTION_COLORS.get(num, "default")
            # セクション間に divider
            if prev_was_h2:
                blocks.append({"object":"block","type":"divider","divider":{}})
            blocks.append({
                "object": "block", "type": "heading_2",
                "heading_2": {
                    "rich_text": _rt(f"{icon}  {heading_text}"),
                    "color": f"{color}_background",
                },
            })
            prev_was_h2 = True

        # H3
        elif line.startswith("### "):
            heading_text = line[4:].strip()
            blocks.append({"object":"block","type":"heading_3",
                            "heading_3":{"rich_text":_rt(heading_text)}})
            prev_was_h2 = False

        # 引用ブロック
        elif s.startswith("> "):
            blocks.append({"object":"block","type":"quote",
                            "quote":{"rich_text":_rt(s[2:])}})

        # 箇条書き
        elif s.startswith("- ") or s.startswith("* "):
            blocks.append({"object":"block","type":"bulleted_list_item",
                            "bulleted_list_item":{"rich_text":_rt(s[2:])}})

        # 番号付きリスト
        elif len(s) > 2 and s[0].isdigit() and s[1] in ".)" and s[2] == " ":
            blocks.append({"object":"block","type":"numbered_list_item",
                            "numbered_list_item":{"rich_text":_rt(s[3:])}})

        # 区切り線
        elif s in ("---","***","___"):
            blocks.append({"object":"block","type":"divider","divider":{}})

        # 段落
        else:
            blocks.append({"object":"block","type":"paragraph",
                            "paragraph":{"rich_text":_rt(s)}})

    _flush_table()
    _flush_code()
    return blocks

# ── 実行 ─────────────────────────────────────────────────────────────────
if not NOTION_TOKEN or not NOTION_PAGE_ID:
    print("⚠ NOTION_TOKEN と NOTION_PAGE_ID が未設定です")

elif "_report_text" not in dir() or not _report_text:
    print("⚠ LLM レポートが未生成です。先に Cell M3 を実行してください")

else:
    print("Notion に接続中...")
    try:
        _nc  = _NC(auth=NOTION_TOKEN)
        _me  = _nc.users.me()
        print(f"✓ 認証成功  ({_me.get('name', 'unknown')})\n")

        _title  = f"特徴量分析レポート {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}  [{LLM_MODEL}]"
        _blocks = _md_to_blocks(_report_text)

        # ── ページ冒頭に ROI サマリー callout ────────────────────────────
        _rs = feature_analysis_json.get("roi_stats", {})
        if _rs and "roi_top1_pct" in _rs:
            _n_races_str = f"{int(_rs['n_races']):,}" if _rs.get('n_races') else "?"
            _roi_line = (
                f"単勝的中: {float(_rs['top1_win_rate'])*100:.1f}%　"
                f"複勝的中: {float(_rs['top1_show_rate'])*100:.1f}%　"
                f"Top3包含: {float(_rs['top3_hit_rate'])*100:.1f}%　"
                f"ROI: {float(_rs['roi_top1_pct']):+.1f}%　"
                f"テストレース数: {_n_races_str}件"
            )
            _blocks.insert(0, {
                "object": "block", "type": "callout",
                "callout": {
                    "rich_text": [{"type":"text","text":{"content":_roi_line}}],
                    "icon": {"type":"emoji","emoji":"📊"},
                    "color": "blue_background",
                },
            })
            # メタ情報 callout
            _meta = feature_analysis_json.get("meta", {})
            _n_train = _meta.get('n_train_rows')
            _n_train_str = f"{int(_n_train):,}" if isinstance(_n_train, (int, float)) else str(_n_train or '?')
            _meta_line = (
                f"モデル: {LLM_MODEL}　"
                f"特徴量数: {_meta.get('n_features','?')}　"
                f"学習行数: {_n_train_str}　"
                f"Spearman: {_meta.get('spearman','?')}"
            )
            _blocks.insert(1, {
                "object": "block", "type": "callout",
                "callout": {
                    "rich_text": [{"type":"text","text":{"content":_meta_line}}],
                    "icon": {"type":"emoji","emoji":"🤖"},
                    "color": "gray_background",
                },
            })

        # ── ページ作成 (100ブロックずつ) ─────────────────────────────────
        _CHUNK = 100
        _new_page = _nc.pages.create(
            parent={"page_id": _page_id},
            properties={"title": {"title": [{"type":"text","text":{"content":_title}}]}},
            children=_blocks[:_CHUNK],
        )
        _pid = _new_page["id"]

        for _i in range(_CHUNK, len(_blocks), _CHUNK):
            _nc.blocks.children.append(block_id=_pid, children=_blocks[_i:_i+_CHUNK])

        _notion_url = f"https://notion.so/{_pid.replace('-','')}"
        print(f"✓ ページ作成完了  ({len(_blocks)} ブロック)")
        print(f"  タイトル: {_title}")
        print(f"  URL: {_notion_url}")

    except Exception as _ne:
        print(f"❌ Notion エラー: {_ne}")

In [ ]:
_corrs_check = feature_analysis_json.get("high_corr_pairs", [])
print("count:", len(_corrs_check))
if _corrs_check:
    print("keys:", list(_corrs_check[0].keys()))
    print("sample[0]:", _corrs_check[0])
    print("sample[1]:", _corrs_check[1])

In [ ]:
_cp = feature_analysis_json.get("high_corr_pairs", [])
print(f"count: {len(_cp)}, keys: {list(_cp[0].keys()) if _cp else 'empty'}")
if _cp: print("sample:", _cp[0])